In [78]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')
# Enveda CASMI 2026 — Kaggle Notebook starter
from pathlib import Path
import glob
import numpy as np
import pandas as pd

# Cell 1: locate competition files
test_paths = glob.glob("/kaggle/input/**/test.parquet", recursive=True)
print("Found test files:", test_paths)
if not test_paths:
    raise FileNotFoundError("Attach the competition dataset to this Kaggle Notebook.")
DATA = Path(test_paths[0]).parent
TRAIN_PATH, TEST_PATH = DATA/"train.parquet", DATA/"test.parquet"
SAMPLE_PATH = DATA/"sample_submission.csv"

# Cell 2: inspect test and submission schema
test = pd.read_parquet(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)
print("Test shape:", test.shape)
print("Test columns:", test.columns.tolist())
print("Spectra per molecule summary:")
print(test.groupby("molecule_id").size().describe())
print("Sample submission:", sample.shape, sample.columns.tolist())
print("Adduct counts:")
print(test["adduct"].value_counts(dropna=False))
print("Ionization mode counts:")
print(test["ionization_mode"].value_counts(dropna=False))

# Cell 3: read a small, deterministic slice of training data to smoke-test the pipeline.
# Once this works, increase the slice or build a vectorized index over the full dataset.
TRAIN_ROWS_FOR_SMOKE_TEST = 50000
# Read only a few Parquet row groups instead of materializing the entire ~3 GB training file.
import pyarrow.parquet as pq
pf = pq.ParquetFile(TRAIN_PATH)
print("Train row groups:", pf.num_row_groups, "| total rows:", pf.metadata.num_rows)
needed = ["precursor_mz", "adduct", "normalized_smiles", "inchikey14"]
if "ingest_lib" in pf.schema.names:
    needed.append("ingest_lib")
chunks, rows_read = [], 0
for rg in range(pf.num_row_groups):
    chunk = pf.read_row_group(rg, columns=needed).to_pandas()
    chunks.append(chunk)
    rows_read += len(chunk)
    if rows_read >= TRAIN_ROWS_FOR_SMOKE_TEST:
        break
train_small = pd.concat(chunks, ignore_index=True).head(TRAIN_ROWS_FOR_SMOKE_TEST)
print("Loaded training smoke-test rows:", train_small.shape)
print("Top source libraries:")
if "ingest_lib" in train_small:
    print(train_small["ingest_lib"].value_counts(dropna=False).head(10))
print("Train sample:")
print(train_small[["precursor_mz", "adduct", "normalized_smiles", "inchikey14"]].head())

# Cell 4: small precursor-mass retrieval baseline.
# This ranks training structures with nearby precursor m/z. It is deliberately a simple
# benchmark to validate output plumbing, not a final spectral similarity model.
train_small = train_small.dropna(subset=["precursor_mz", "normalized_smiles"])
mass = train_small["precursor_mz"].to_numpy(dtype=float)
smiles = train_small["normalized_smiles"].astype(str).to_numpy()
keys = train_small["inchikey14"].astype(str).to_numpy() if "inchikey14" in train_small else smiles

# Candidate ranking uses nearest precursor m/z, with duplicate structures deduplicated.
# Spectrum-aware scoring is the next upgrade.
submission_rows = []
for n, molecule_id in enumerate(sample["molecule_id"]):
    group = test[test["molecule_id"] == molecule_id]
    scores = {}
    for row in group.itertuples(index=False):
        distance = np.abs(mass - float(row.precursor_mz))
        nearby = np.flatnonzero(distance <= 0.05)
        if len(nearby) == 0:
            nearby = np.argsort(distance)[:100]
        # Keep each structure's best mass match across all spectra for this molecule.
        for j in nearby:
            smi = smiles[j]
            score = -float(distance[j])
            if smi not in scores or score > scores[smi]:
                scores[smi] = score
    ranked = [s for s, _ in sorted(scores.items(), key=lambda kv: kv[1], reverse=True)[:25]]
    if not ranked:
        ranked = ["CCO"]  # valid-format fallback only; should be rare and is not a chemistry prediction
    submission_rows.append({"molecule_id": molecule_id, "smiles": ";".join(ranked)})
    if (n + 1) % 50 == 0:
        print(f"Processed {n+1}/{len(sample)} molecules")

submission = pd.DataFrame(submission_rows)
assert submission.columns.tolist() == ["molecule_id", "smiles"]
assert submission["molecule_id"].is_unique
assert submission["smiles"].notna().all() and (submission["smiles"].str.len() > 0).all()
assert (submission["smiles"].str.split(";").str.len() <= 25).all()
out = "/kaggle/working/submission_baseline.csv"
submission.to_csv(out, index=False)
print("Wrote", out, "shape=", submission.shape)
print(submission.head().to_string(index=False))


/kaggle/input/datasets/denisketer/enveda-casmi-2026/sample_submission.csv
/kaggle/input/datasets/denisketer/enveda-casmi-2026/train.parquet
/kaggle/input/datasets/denisketer/enveda-casmi-2026/test.parquet
Found test files: ['/kaggle/input/datasets/denisketer/enveda-casmi-2026/test.parquet']
Test shape: (1213, 12)
Test columns: ['molecule_id', 'spectrum_id', 'ms2_mzs', 'ms2_normalized_intensities', 'base_peak_intensity', 'adduct', 'ionization_mode', 'instrument_type', 'precursor_mz', 'collision_energy_orig', 'collision_energy_ev', 'collision_energy_orig_units']
Spectra per molecule summary:
count    400.000000
mean       3.032500
std        1.382469
min        1.000000
25%        2.000000
50%        3.000000
75%        4.000000
max        9.000000
dtype: float64
Sample submission: (400, 2) ['molecule_id', 'smiles']
Adduct counts:
adduct
[M+H]+          959
[M-H]-          193
[M+CH2O2-H]-     31
[M+Na]+          22
[M+NH4]+          4
[M+K]+            2
[M+Cl]-           2
Name: count,

In [79]:
import pandas as pd
from pathlib import Path

submission_path = Path("/kaggle/working/submission_baseline.csv")
sample_path = Path(
    "/kaggle/input/datasets/denisketer/enveda-casmi-2026/sample_submission.csv"
)

assert submission_path.exists(), "Submission file not found!"

sub = pd.read_csv(submission_path)
sample = pd.read_csv(sample_path)

print("Submission shape:", sub.shape)
print("Expected shape:", sample.shape)
print("Columns:", sub.columns.tolist())
print("Unique molecules:", sub["molecule_id"].nunique())
print("Missing values:\n", sub.isna().sum())

# Check whether each row has up to 25 candidate SMILES
candidate_counts = sub["smiles"].fillna("").apply(
    lambda x: len([s for s in x.split(";") if s.strip()])
)
print("\nCandidate counts:")
print(candidate_counts.describe())
print("Rows with zero candidates:", (candidate_counts == 0).sum())

assert sub.columns.tolist() == sample.columns.tolist()
assert sub["molecule_id"].is_unique
assert len(sub) == len(sample)
assert (candidate_counts <= 25).all()

print("\nBasic submission checks passed.")

Submission shape: (400, 2)
Expected shape: (400, 2)
Columns: ['molecule_id', 'smiles']
Unique molecules: 400
Missing values:
 molecule_id    0
smiles         0
dtype: int64

Candidate counts:
count    400.000000
mean      22.797500
std        5.392048
min        1.000000
25%       25.000000
50%       25.000000
75%       25.000000
max       25.000000
Name: smiles, dtype: float64
Rows with zero candidates: 0

Basic submission checks passed.


In [80]:
import pyarrow.parquet as pq
from pathlib import Path

train_path = Path(
    "/kaggle/input/datasets/denisketer/enveda-casmi-2026/train.parquet"
)

parquet_file = pq.ParquetFile(train_path)

print("Training rows:", parquet_file.metadata.num_rows)
print("Row groups:", parquet_file.num_row_groups)
print("\nTraining schema:")
print(parquet_file.schema)

Training rows: 2539608
Row groups: 21

Training schema:
required group field_id=-1 schema {
  optional binary field_id=-1 ingest_lib (String);
  optional binary field_id=-1 normalized_smiles (String);
  optional binary field_id=-1 inchikey (String);
  optional binary field_id=-1 inchikey14 (String);
  optional binary field_id=-1 molecular_formula (String);
  optional binary field_id=-1 ionization_mode (String);
  optional binary field_id=-1 instrument_type (String);
  optional binary field_id=-1 adduct (String);
  optional binary field_id=-1 adduct_orig (String);
  optional double field_id=-1 precursor_mz;
  optional double field_id=-1 precursor_error_ppm;
  optional group field_id=-1 ms2_mzs (List) {
    repeated group field_id=-1 list {
      optional double field_id=-1 element;
    }
  }
  optional group field_id=-1 ms2_normalized_intensities (List) {
    repeated group field_id=-1 list {
      optional double field_id=-1 element;
    }
  }
  optional int64 field_id=-1 num_peaks;
  

In [81]:
import pyarrow.parquet as pq
import pandas as pd
from pathlib import Path

train_path = Path(
    "/kaggle/input/datasets/denisketer/enveda-casmi-2026/train.parquet"
)

# Read only the columns needed for an initial inspection
columns = [
    "normalized_smiles",
    "inchikey14",
    "molecular_formula",
    "precursor_mz",
    "adduct",
    "ionization_mode",
    "ingest_lib",
]

pf = pq.ParquetFile(train_path)

total_rows = 0
missing_counts = {c: 0 for c in columns}
adduct_counts = {}
library_counts = {}
unique_keys = set()
mass_min = float("inf")
mass_max = float("-inf")

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=columns
):
    df = batch.to_pandas()
    total_rows += len(df)

    for col in columns:
        missing_counts[col] += int(df[col].isna().sum())

    # Count common adducts and source libraries
    for key, count in df["adduct"].value_counts(
        dropna=False
    ).items():
        adduct_counts[key] = adduct_counts.get(key, 0) + int(count)

    for key, count in df["ingest_lib"].value_counts(
        dropna=False
    ).items():
        library_counts[key] = library_counts.get(key, 0) + int(count)

    # Count distinct labelled structures
    valid_keys = df["inchikey14"].dropna().unique()
    unique_keys.update(valid_keys)

    # Track the precursor mass range
    masses = df["precursor_mz"].dropna()
    if not masses.empty:
        mass_min = min(mass_min, masses.min())
        mass_max = max(mass_max, masses.max())

print("Rows inspected:", total_rows)
print("Distinct InChIKey14 labels:", len(unique_keys))
print("Precursor m/z range:", mass_min, "to", mass_max)

print("\nMissing values:")
print(pd.Series(missing_counts).sort_values(ascending=False))

print("\nMost common adducts:")
print(pd.Series(adduct_counts).sort_values(ascending=False).head(10))

print("\nSource libraries:")
print(pd.Series(library_counts).sort_values(ascending=False))

Rows inspected: 2539608
Distinct InChIKey14 labels: 275810
Precursor m/z range: 2.0 to 2430333.0

Missing values:
normalized_smiles    0
inchikey14           0
molecular_formula    0
precursor_mz         0
adduct               0
ionization_mode      0
ingest_lib           0
dtype: int64

Most common adducts:
[M+H]+          1324585
[M-H]-           465688
[2M+Na]+         190033
[M+Na]+          178092
[2M+H]+           92041
[M+CH2O2-H]-      83188
[2M-H]-           56172
[M+NH4]+          53452
[M+Cl]-           27511
[M-H2O+H]+        21682
dtype: int64

Source libraries:
enveda-180            1153785
pluskal_ms2            527581
riken                  347171
gnps                   220849
massbank               101727
mona                    92416
spectraverse            50933
msdial                  40765
drug_plus                2545
enveda-np-examples       1184
masaryk                   652
dtype: int64


In [82]:
import pyarrow.parquet as pq
import pandas as pd
import numpy as np

train_path = (
    "/kaggle/input/datasets/denisketer/"
    "enveda-casmi-2026/train.parquet"
)

pf = pq.ParquetFile(train_path)

batches = []

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=["precursor_mz", "adduct", "ingest_lib"]
):
    batches.append(batch.to_pandas())

mass_df = pd.concat(batches, ignore_index=True)

print("Overall precursor m/z quantiles:")
print(
    mass_df["precursor_mz"].quantile(
        [0, 0.001, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999, 1]
    )
)

print("\nLowest 10 precursor masses:")
print(
    mass_df.nsmallest(10, "precursor_mz").to_string(index=False)
)

print("\nHighest 10 precursor masses:")
print(
    mass_df.nlargest(10, "precursor_mz").to_string(index=False)
)

print("\nMass quantiles by common adduct:")
for adduct, group in mass_df.groupby("adduct"):
    if len(group) >= 1000:
        print(
            adduct,
            "count =", len(group),
            "median =", round(group["precursor_mz"].median(), 3),
            "99th percentile =", round(
                group["precursor_mz"].quantile(0.99), 3
            )
        )

print("\nRows below m/z 50:", (mass_df["precursor_mz"] < 50).sum())
print("Rows above m/z 2000:", (mass_df["precursor_mz"] > 2000).sum())

Overall precursor m/z quantiles:
0.000    2.000000e+00
0.001    9.497000e+01
0.010    1.370710e+02
0.050    2.040655e+02
0.250    3.081410e+02
0.500    3.501088e+02
0.750    4.892142e+02
0.950    7.754660e+02
0.990    9.763292e+02
0.999    1.384516e+03
1.000    2.430333e+06
Name: precursor_mz, dtype: float64

Lowest 10 precursor masses:
 precursor_mz   adduct ingest_lib
       2.0000 [M+2H]2+       gnps
       2.0000 [M+2H]2+       gnps
       2.0000 [M+2H]2+       gnps
      32.0495   [M+H]+  drug_plus
      32.0500   [M+H]+       gnps
      32.0500   [M+H]+       gnps
      32.0500   [M+H]+       gnps
      33.0340   [M+H]+       gnps
      33.0340   [M+H]+       gnps
      34.0293   [M+H]+       gnps

Highest 10 precursor masses:
 precursor_mz  adduct ingest_lib
 2430333.0000  [M-H]-       mona
    3355.6000  [M+H]+       gnps
    2679.4869  [M+H]+     msdial
    2679.4800  [M+H]+      riken
    2647.5000  [M+H]+      riken
    2647.4970  [M+H]+     msdial
    2465.1982  [M+H]+     

In [83]:
import pyarrow.parquet as pq
import pandas as pd

train_path = (
    "/kaggle/input/datasets/denisketer/"
    "enveda-casmi-2026/train.parquet"
)

pf = pq.ParquetFile(train_path)

columns = [
    "precursor_mz",
    "adduct",
    "ingest_lib",
    "normalized_smiles",
    "molecular_formula",
    "inchikey14",
    "precursor_error_ppm",
    "num_peaks",
]

outliers = []

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=columns
):
    df = batch.to_pandas()

    unusual = df[
        (df["precursor_mz"] < 50) |
        (df["precursor_mz"] > 2000)
    ]

    if not unusual.empty:
        outliers.append(unusual)

outliers = pd.concat(outliers, ignore_index=True)

print("Number of unusual records:", len(outliers))

print(
    outliers[
        [
            "precursor_mz",
            "adduct",
            "ingest_lib",
            "molecular_formula",
            "precursor_error_ppm",
            "num_peaks",
            "inchikey14",
        ]
    ].sort_values("precursor_mz").to_string(index=False)
)

Number of unusual records: 77
 precursor_mz   adduct ingest_lib molecular_formula  precursor_error_ppm  num_peaks     inchikey14
       2.0000 [M+2H]2+       gnps     C99H140N20O17         4.697715e+08       1676 ZWCXYZRRTRDGQE
       2.0000 [M+2H]2+       gnps     C99H140N20O17         4.697715e+08       2023 ZWCXYZRRTRDGQE
       2.0000 [M+2H]2+       gnps     C99H140N20O17         4.697715e+08       1660 ZWCXYZRRTRDGQE
      32.0495   [M+H]+  drug_plus              CH5N         1.635588e+01          3 BAVYZALUXZFZLV
      32.0500   [M+H]+       gnps              CH5N         7.550019e-01          3 BAVYZALUXZFZLV
      32.0500   [M+H]+       gnps              CH5N         7.550019e-01          4 BAVYZALUXZFZLV
      32.0500   [M+H]+       gnps              CH5N         7.550019e-01          4 BAVYZALUXZFZLV
      33.0340   [M+H]+       gnps              CH4O         1.204236e+00          1 OKKJLVBELUTLKV
      33.0340   [M+H]+       gnps              CH4O         1.204236e+00       

In [84]:
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, rdMolDescriptors

    print("RDKit is available.")

    mol = Chem.MolFromSmiles("CCO")
    print("Test molecule: ethanol")
    print("Molecular formula:", rdMolDescriptors.CalcMolFormula(mol))
    print("Molecular weight:", round(Descriptors.MolWt(mol), 4))

except ImportError:
    print("RDKit is not installed in this environment.")

RDKit is available.
Test molecule: ethanol
Molecular formula: C2H6O
Molecular weight: 46.069


In [85]:
%pip install rdkit

Note: you may need to restart the kernel to use updated packages.


In [86]:
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

mol = Chem.MolFromSmiles("CCO")

print("RDKit version:", __import__("rdkit").__version__)
print("SMILES valid:", mol is not None)
print("Molecular formula:", rdMolDescriptors.CalcMolFormula(mol))
print("Exact molecular mass:", round(Descriptors.ExactMolWt(mol), 5))

RDKit version: 2026.09.1
SMILES valid: True
Molecular formula: C2H6O
Exact molecular mass: 46.04186


In [87]:
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, rdMolDescriptors

    print("RDKit is available.")

    mol = Chem.MolFromSmiles("CCO")
    print("Test molecule: ethanol")
    print("Molecular formula:", rdMolDescriptors.CalcMolFormula(mol))
    print("Molecular weight:", round(Descriptors.MolWt(mol), 4))

except ImportError:
    print("RDKit is not installed in this environment.")

RDKit is available.
Test molecule: ethanol
Molecular formula: C2H6O
Molecular weight: 46.069


In [88]:
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

mol = Chem.MolFromSmiles("CCO")

print("Molecular formula:", rdMolDescriptors.CalcMolFormula(mol))
print("Average molecular weight:", round(Descriptors.MolWt(mol), 5))
print("Exact molecular mass:", round(Descriptors.ExactMolWt(mol), 5))

Molecular formula: C2H6O
Average molecular weight: 46.069
Exact molecular mass: 46.04186


In [89]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

DATA_DIR = "/kaggle/input/datasets/denisketer/enveda-casmi-2026"

# Read only the columns needed to construct the structure library.
train = pd.read_parquet(
    f"{DATA_DIR}/train.parquet",
    columns=[
        "normalized_smiles",
        "inchikey14",
        "molecular_formula",
    ],
)

# One row per InChIKey14, retaining a representative SMILES.
library = train.drop_duplicates("inchikey14").copy()
library = library.dropna(subset=["normalized_smiles", "inchikey14"])

def describe_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return {
        "exact_mass": Descriptors.ExactMolWt(mol),
        "rdkit_formula": rdMolDescriptors.CalcMolFormula(mol),
    }

descriptions = library["normalized_smiles"].map(describe_smiles)
valid = descriptions.notna()

library = library.loc[valid].copy()
library["exact_mass"] = [
    item["exact_mass"] for item in descriptions.loc[valid]
]
library["rdkit_formula"] = [
    item["rdkit_formula"] for item in descriptions.loc[valid]
]

library = library.reset_index(drop=True)

print("Unique training structures:", train["inchikey14"].nunique())
print("Valid candidate structures:", len(library))
print("Missing exact masses:", library["exact_mass"].isna().sum())
display(library.head())

Unique training structures: 275810
Valid candidate structures: 275810
Missing exact masses: 0


,normalized_smiles,inchikey14,molecular_formula,exact_mass,rdkit_formula
0,O=C1NC(=O)c2cc(Nc3ccccc3)c(Nc3ccccc3)cc21,AAALVYBICLMAMA,C20H15N3O2,329.116427,C20H15N3O2
1,C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1,AAKJLRGGTJKAMG,C22H23N3O4,393.168856,C22H23N3O4
2,CC1(C)CCC(C)(C)c2cc(C(O)C(O)=Nc3ccc(C(=O)O)cc3...,AANFHDFOMFRLLR,C23H26FNO4,399.184587,C23H26FNO4
3,CN1C(=O)CN=C(c2ccccc2)c2cc(Cl)ccc21,AAOVKJBEBIDNHE,C16H13ClN2O,284.071641,C16H13ClN2O
4,NC(=O)C1=CN(C2OC(COP(=O)(O)OP(=O)(O)OCC3OC(n4c...,ACFIXJIJDZMPPO,C21H30N7O17P3,745.091102,C21H30N7O17P3


In [90]:
import pandas as pd

DATA_DIR = "/kaggle/input/datasets/denisketer/enveda-casmi-2026"

train_meta = pd.read_parquet(
    f"{DATA_DIR}/train.parquet",
    columns=[
        "inchikey14",
        "precursor_mz",
        "adduct",
        "ionization_mode",
        "precursor_error_ppm",
    ],
)

print("Rows:", len(train_meta))
print("\nMost common adducts:")
display(train_meta["adduct"].value_counts().head(20).to_frame("count"))

print("\nIonization modes:")
display(train_meta["ionization_mode"].value_counts(dropna=False).to_frame("count"))

print("\nPrecursor error (ppm) summary:")
display(train_meta["precursor_error_ppm"].describe(
    percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
).to_frame("precursor_error_ppm"))

Rows: 2539608

Most common adducts:


,count
adduct,
[M+H]+,1324585
[M-H]-,465688
[2M+Na]+,190033
[M+Na]+,178092
[2M+H]+,92041
[M+CH2O2-H]-,83188
[2M-H]-,56172
[M+NH4]+,53452
[M+Cl]-,27511



Ionization modes:


,count
ionization_mode,
positive,1899640
negative,639968



Precursor error (ppm) summary:


,precursor_error_ppm
count,2.535322e+06
mean,1.845726e+03
std,5.117525e+05
min,-2.106149e+01
1%,3.002888e-02
5%,1.544535e-01
50%,1.512113e+00
95%,1.718147e+01
99%,2.884419e+03
max,4.697715e+08


In [91]:
test_meta = pd.read_parquet(
    f"{DATA_DIR}/test.parquet",
    columns=[
        "molecule_id",
        "spectrum_id",
        "precursor_mz",
        "adduct",
        "ionization_mode",
    ],
)

print("Test spectra:", len(test_meta))
print("Test molecules:", test_meta["molecule_id"].nunique())
print("\nTest adducts:")
display(test_meta["adduct"].value_counts(dropna=False).to_frame("count"))

print("\nExample test spectra:")
display(test_meta.head(10))

Test spectra: 1213
Test molecules: 400

Test adducts:


,count
adduct,
[M+H]+,959
[M-H]-,193
[M+CH2O2-H]-,31
[M+Na]+,22
[M+NH4]+,4
[M+K]+,2
[M+Cl]-,2



Example test spectra:


,molecule_id,spectrum_id,precursor_mz,adduct,ionization_mode
0,m_005e53,s_4d19d522,375.0709,[M+H]+,positive
1,m_005e53,s_5b6e28ff,375.0709,[M+H]+,positive
2,m_005e53,s_6e514292,375.0709,[M+H]+,positive
3,m_005e53,s_855ae19c,373.0562,[M-H]-,negative
4,m_006153,s_3c74e9fe,342.1569,[M+H]+,positive
5,m_006153,s_54245904,340.1416,[M-H]-,negative
6,m_006153,s_59a80abf,340.1416,[M-H]-,negative
7,m_006153,s_7151fa86,342.1569,[M+H]+,positive
8,m_006153,s_86472581,340.1416,[M-H]-,negative
9,m_006153,s_cb15f0ac,342.1568,[M+H]+,positive


In [92]:
import pandas as pd

DATA_DIR = "/kaggle/input/datasets/denisketer/enveda-casmi-2026"

test_meta = pd.read_parquet(
    f"{DATA_DIR}/test.parquet",
    columns=[
        "molecule_id",
        "spectrum_id",
        "precursor_mz",
        "adduct",
        "ionization_mode",
    ],
)

print("Test spectra:", len(test_meta))
print("Test molecules:", test_meta["molecule_id"].nunique())

print("\nTest adducts:")
display(test_meta["adduct"].value_counts(dropna=False).to_frame("count"))

print("\nIonization modes:")
display(test_meta["ionization_mode"].value_counts(dropna=False).to_frame("count"))

print("\nExample spectra:")
display(test_meta.head(10))


Test spectra: 1213
Test molecules: 400

Test adducts:


,count
adduct,
[M+H]+,959
[M-H]-,193
[M+CH2O2-H]-,31
[M+Na]+,22
[M+NH4]+,4
[M+K]+,2
[M+Cl]-,2



Ionization modes:


,count
ionization_mode,
positive,987
negative,226



Example spectra:


,molecule_id,spectrum_id,precursor_mz,adduct,ionization_mode
0,m_005e53,s_4d19d522,375.0709,[M+H]+,positive
1,m_005e53,s_5b6e28ff,375.0709,[M+H]+,positive
2,m_005e53,s_6e514292,375.0709,[M+H]+,positive
3,m_005e53,s_855ae19c,373.0562,[M-H]-,negative
4,m_006153,s_3c74e9fe,342.1569,[M+H]+,positive
5,m_006153,s_54245904,340.1416,[M-H]-,negative
6,m_006153,s_59a80abf,340.1416,[M-H]-,negative
7,m_006153,s_7151fa86,342.1569,[M+H]+,positive
8,m_006153,s_86472581,340.1416,[M-H]-,negative
9,m_006153,s_cb15f0ac,342.1568,[M+H]+,positive


In [93]:
import numpy as np
import pandas as pd
from collections import defaultdict

# Exact mass offsets for the adducts observed in the test set.
# The offsets convert neutral molecular exact mass to ion m/z.
PROTON = 1.0072764666

ADDUCT_OFFSETS = {
    "[M+H]+":       PROTON,
    "[M-H]-":      -PROTON,
    "[M+CH2O2-H]-": 44.998201,
    "[M+Na]+":      22.989218,
    "[M+NH4]+":     18.033823,
    "[M+K]+":       38.963158,
    "[M+Cl]-":      34.969402,
}

# Prepare the candidate mass catalog for each supported adduct.
candidate_catalog = {}

for adduct, offset in ADDUCT_OFFSETS.items():
    masses = library["exact_mass"].to_numpy(dtype=np.float64) + offset
    order = np.argsort(masses)
    candidate_catalog[adduct] = (
        masses[order],
        library["inchikey14"].to_numpy()[order],
        library["normalized_smiles"].to_numpy()[order],
    )

# Build a validation sample from ordinary precursor measurements.
# This is a retrieval sanity check, not a fully leakage-proof estimate
# of performance on unseen chemical structures.
val = train_meta[
    train_meta["adduct"].isin(ADDUCT_OFFSETS)
    & train_meta["precursor_mz"].between(50, 2000)
    & train_meta["precursor_mz"].notna()
    & train_meta["inchikey14"].notna()
].sample(n=min(5000, len(train_meta[
    train_meta["adduct"].isin(ADDUCT_OFFSETS)
    & train_meta["precursor_mz"].between(50, 2000)
    & train_meta["precursor_mz"].notna()
    & train_meta["inchikey14"].notna()
])), random_state=42)

TOLERANCE_PPM = 20
reciprocal_ranks = []
covered = 0
evaluated = 0

for row in val.itertuples(index=False):
    adduct = row.adduct
    observed_mz = float(row.precursor_mz)
    target_key = row.inchikey14

    masses, keys, smiles = candidate_catalog[adduct]
    tolerance = observed_mz * TOLERANCE_PPM / 1e6

    left = np.searchsorted(masses, observed_mz - tolerance, side="left")
    right = np.searchsorted(masses, observed_mz + tolerance, side="right")

    if left == right:
        reciprocal_ranks.append(0.0)
        evaluated += 1
        continue

    local_masses = masses[left:right]
    local_keys = keys[left:right]

    errors_ppm = np.abs(local_masses - observed_mz) / observed_mz * 1e6
    order = np.argsort(errors_ppm, kind="stable")
    ranked_keys = local_keys[order][:25]

    matches = np.flatnonzero(ranked_keys == target_key)
    if len(matches):
        rank = int(matches[0]) + 1
        reciprocal_ranks.append(1.0 / rank)
        covered += 1
    else:
        reciprocal_ranks.append(0.0)

    evaluated += 1

print(f"Validation spectra evaluated: {evaluated:,}")
print(f"Target found in top 25: {covered:,} ({covered / evaluated:.1%})")
print(f"Mass-retrieval MRR@25: {np.mean(reciprocal_ranks):.5f}")
print(f"Mass tolerance: ±{TOLERANCE_PPM} ppm")
print(f"Candidate structures: {len(library):,}")

Validation spectra evaluated: 5,000
Target found in top 25: 3,693 (73.9%)
Mass-retrieval MRR@25: 0.31661
Mass tolerance: ±20 ppm
Candidate structures: 275,810


In [94]:
# Uses the existing train DataFrame if it is still in memory.
# Otherwise, read only the necessary columns.

import pandas as pd

DATA_DIR = "/kaggle/input/datasets/denisketer/enveda-casmi-2026"

train_check = pd.read_parquet(
    f"{DATA_DIR}/train.parquet",
    columns=["inchikey14", "normalized_smiles"],
)

smiles_per_key = (
    train_check.groupby("inchikey14")["normalized_smiles"]
    .nunique()
)

print("Unique InChIKey14:", train_check["inchikey14"].nunique())
print("Keys with multiple SMILES:", (smiles_per_key > 1).sum())
print("Maximum SMILES per key:", smiles_per_key.max())

spectra_per_key = train_check.groupby("inchikey14").size()

print("\nSpectra per structure:")
print(spectra_per_key.describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.99]
))

Unique InChIKey14: 275810
Keys with multiple SMILES: 1612
Maximum SMILES per key: 5

Spectra per structure:
count    275810.000000
mean          9.207817
std          21.069560
min           1.000000
25%           4.000000
50%           6.000000
75%           9.000000
90%          14.000000
99%          79.000000
max        1468.000000
dtype: float64


In [101]:

import numpy as np
import pandas as pd

DATA_DIR = "/kaggle/input/datasets/denisketer/enveda-casmi-2026"

# Use one representative SMILES per InChIKey14.
structures = (
    train_check[["inchikey14", "normalized_smiles"]]
    .dropna()
    .drop_duplicates(["inchikey14", "normalized_smiles"])
    .sort_values(["inchikey14", "normalized_smiles"])
    .drop_duplicates("inchikey14")
    .reset_index(drop=True)
)

# Deterministic split by structure identity.
rng = np.random.default_rng(42)
indices = rng.permutation(len(structures))
n_val = int(round(0.10 * len(structures)))

val_keys = set(structures.iloc[indices[:n_val]]["inchikey14"])
train_keys = set(structures.iloc[indices[n_val:]]["inchikey14"])

assert val_keys.isdisjoint(train_keys)

print("Total structures:", len(structures))
print("Training structures:", len(train_keys))
print("Validation structures:", len(val_keys))
print("Overlap:", len(train_keys & val_keys))

# All spectra belonging to validation structures are held out.
train_spectra = train_meta[train_meta["inchikey14"].isin(train_keys)].copy()
val_spectra = train_meta[train_meta["inchikey14"].isin(val_keys)].copy()

print("\nTraining spectra:", len(train_spectra))
print("Validation spectra:", len(val_spectra))
print("Validation structures represented:", val_spectra["inchikey14"].nunique())


Total structures: 275810
Training structures: 248229
Validation structures: 27581
Overlap: 0

Training spectra: 2285230
Validation spectra: 254378
Validation structures represented: 27581


In [ ]:

import pandas as pd
import numpy as np

DATA_DIR = "/kaggle/input/datasets/denisketer/enveda-casmi-2026"

spectra = pd.read_parquet(
    f"{DATA_DIR}/train.parquet",
    columns=[
        "inchikey14",
        "normalized_smiles",
        "precursor_mz",
        "adduct",
        "ionization_mode",
        "ms2_mzs",
        "ms2_normalized_intensities",
    ],
)

rng = np.random.default_rng(42)

# Shuffle rows within each structure deterministically.
spectra["_random"] = rng.random(len(spectra))
spectra = spectra.sort_values(
    ["inchikey14", "_random"]
).reset_index(drop=True)

# Reserve approximately 20% of each structure's spectra,
# but never remove the final reference spectrum.
group_sizes = spectra.groupby("inchikey14")["inchikey14"].transform("size")
group_position = spectra.groupby("inchikey14").cumcount()
n_holdout = np.floor(group_sizes * 0.20).astype(int)
n_holdout = np.minimum(n_holdout, group_sizes - 1)

is_validation = group_position < n_holdout

validation_spectra = spectra.loc[is_validation].drop(
    columns="_random"
).copy()

reference_spectra = spectra.loc[~is_validation].drop(
    columns="_random"
).copy()

assert set(validation_spectra["inchikey14"]).issubset(
    set(reference_spectra["inchikey14"])
)

print("Reference spectra:", len(reference_spectra))
print("Held-out spectra:", len(validation_spectra))
print("Structures in reference:", reference_spectra["inchikey14"].nunique())
print("Structures in validation:", validation_spectra["inchikey14"].nunique())
print("Structure overlap:", len(
    set(reference_spectra["inchikey14"])
    & set(validation_spectra["inchikey14"])
))


In [ ]:
print("Reference columns:", reference_spectra.columns.tolist())

print("\nExample fragment m/z values:")
print(reference_spectra["ms2_mzs"].iloc[0][:10])

print("\nExample normalized intensities:")
print(reference_spectra["ms2_normalized_intensities"].iloc[0][:10])

print("\nReference adduct counts:")
display(reference_spectra["adduct"].value_counts().head(10).to_frame("count"))

print("\nValidation adduct counts:")
display(validation_spectra["adduct"].value_counts().head(10).to_frame("count"))

In [ ]:

import pandas as pd
import numpy as np

DATA_DIR = "/kaggle/input/datasets/denisketer/enveda-casmi-2026"

spectra = pd.read_parquet(
    f"{DATA_DIR}/train.parquet",
    columns=[
        "inchikey14",
        "normalized_smiles",
        "precursor_mz",
        "adduct",
        "ionization_mode",
        "ms2_mzs",
        "ms2_normalized_intensities",
    ],
)

rng = np.random.default_rng(42)
spectra["_random"] = rng.random(len(spectra))
spectra = spectra.sort_values(
    ["inchikey14", "_random"]
).reset_index(drop=True)

group_sizes = spectra.groupby("inchikey14")["inchikey14"].transform("size")
group_position = spectra.groupby("inchikey14").cumcount()
n_holdout = np.floor(group_sizes * 0.20).astype(int)
n_holdout = np.minimum(n_holdout, group_sizes - 1)

is_validation = group_position < n_holdout

validation_spectra = (
    spectra.loc[is_validation]
    .drop(columns="_random")
    .copy()
)

reference_spectra = (
    spectra.loc[~is_validation]
    .drop(columns="_random")
    .copy()
)

print("Reference spectra:", len(reference_spectra))
print("Validation spectra:", len(validation_spectra))
print(
    "Structures represented in both:",
    len(
        set(reference_spectra["inchikey14"])
        & set(validation_spectra["inchikey14"])
    ),
)



In [ ]:

import numpy as np
import pandas as pd
from collections import defaultdict
from sklearn.feature_extraction import DictVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

# Keep the first experiment small and reproducible.
N_QUERY = 2000
FRAGMENT_BIN_WIDTH = 0.02
MIN_FRAGMENT_MZ = 50.0
MAX_FRAGMENT_MZ = 1000.0

rng = np.random.default_rng(42)
query_idx = rng.choice(
    len(validation_spectra),
    size=min(N_QUERY, len(validation_spectra)),
    replace=False,
)
queries = validation_spectra.iloc[query_idx].reset_index(drop=True)

# Convert each spectrum into a sparse dictionary of binned peak intensities.
def spectrum_to_dict(mzs, intensities):
    features = {}
    for mz, intensity in zip(mzs, intensities):
        mz = float(mz)
        intensity = float(intensity)

        if not np.isfinite(mz) or not np.isfinite(intensity):
            continue
        if mz < MIN_FRAGMENT_MZ or mz > MAX_FRAGMENT_MZ or intensity <= 0:
            continue

        bin_id = int(round(mz / FRAGMENT_BIN_WIDTH))
        features[bin_id] = features.get(bin_id, 0.0) + intensity

    # Square-root weighting reduces domination by the base peak.
    return {k: np.sqrt(v) for k, v in features.items()}

print("Query spectra:", len(queries))
print("Reference spectra:", len(reference_spectra))
print("Fragment bin width:", FRAGMENT_BIN_WIDTH)
print("Fragment m/z range:", MIN_FRAGMENT_MZ, "to", MAX_FRAGMENT_MZ)


In [ ]:

# Sample background reference spectra.
N_BACKGROUND = 20_000

background = reference_spectra.sample(
    n=min(N_BACKGROUND, len(reference_spectra)),
    random_state=42,
)

# Add one reference spectrum for each query target, where available.
query_keys = set(queries["inchikey14"])

target_refs = (
    reference_spectra[
        reference_spectra["inchikey14"].isin(query_keys)
    ]
    .groupby("inchikey14", sort=False)
    .head(1)
)

# Deduplicate using scalar columns only; fragment arrays are unhashable.
refs = pd.concat(
    [background, target_refs],
    axis=0,
).drop_duplicates(
    subset=[
        "inchikey14",
        "precursor_mz",
        "adduct",
        "ionization_mode",
    ],
).reset_index(drop=True)

print("Queries:", len(queries))
print("Reference spectra sampled:", len(refs))
print(
    "Query target structures with references:",
    len(query_keys & set(refs["inchikey14"])),
)



In [ ]:

import numpy as np
from sklearn.feature_extraction import DictVectorizer
from sklearn.preprocessing import normalize

def spectrum_to_dict(mzs, intensities):
    features = {}
    for mz, intensity in zip(mzs, intensities):
        mz, intensity = float(mz), float(intensity)
        if not np.isfinite(mz) or not np.isfinite(intensity):
            continue
        if 50 <= mz <= 1000 and intensity > 0:
            bin_id = int(round(mz / 0.02))
            features[bin_id] = features.get(bin_id, 0.0) + intensity
    return {k: np.sqrt(v) for k, v in features.items()}

ref_dicts = [
    spectrum_to_dict(mzs, ints)
    for mzs, ints in zip(
        refs["ms2_mzs"], refs["ms2_normalized_intensities"]
    )
]
query_dicts = [
    spectrum_to_dict(mzs, ints)
    for mzs, ints in zip(
        queries["ms2_mzs"], queries["ms2_normalized_intensities"]
    )
]

vectorizer = DictVectorizer(dtype=np.float32)
R = normalize(vectorizer.fit_transform(ref_dicts), norm="l2")
Q = normalize(vectorizer.transform(query_dicts), norm="l2")

ref_modes = refs["ionization_mode"].fillna("unknown").to_numpy()
ref_keys = refs["inchikey14"].to_numpy()
query_modes = queries["ionization_mode"].fillna("unknown").to_numpy()
query_keys_arr = queries["inchikey14"].to_numpy()

rr = []
top25_hits = 0

for i in range(len(queries)):
    compatible = np.flatnonzero(ref_modes == query_modes[i])
    if len(compatible) == 0:
        rr.append(0.0)
        continue

    sims = (Q[i] @ R[compatible].T).toarray().ravel()

    best_by_key = {}
    for j, score in zip(compatible, sims):
        key = ref_keys[j]
        if score > best_by_key.get(key, -1):
            best_by_key[key] = float(score)

    ranked_keys = [
        key for key, score in sorted(
            best_by_key.items(), key=lambda item: item[1], reverse=True
        )[:25]
    ]

    if query_keys_arr[i] in ranked_keys:
        rank = ranked_keys.index(query_keys_arr[i]) + 1
        rr.append(1.0 / rank)
        top25_hits += 1
    else:
        rr.append(0.0)

print("Queries evaluated:", len(queries))
print("Target in top 25:", f"{top25_hits / len(queries):.1%}")
print("MRR@25:", f"{np.mean(rr):.5f}")



In [ ]:

import numpy as np

ref_modes = refs["ionization_mode"].fillna("unknown").to_numpy()
ref_keys = refs["inchikey14"].to_numpy()
query_modes = queries["ionization_mode"].fillna("unknown").to_numpy()
query_keys_arr = queries["inchikey14"].to_numpy()

rr_fair = []
hits_fair = 0

for i in range(len(queries)):
    # Same ionization mode, but exclude every reference spectrum
    # belonging to the query's true structure.
    compatible = np.flatnonzero(
        (ref_modes == query_modes[i])
        & (ref_keys != query_keys_arr[i])
    )

    # This exclusion is intentionally strict: the correct structure
    # cannot be retrieved from this reference set.
    if len(compatible) == 0:
        rr_fair.append(0.0)
        continue

    sims = (Q[i] @ R[compatible].T).toarray().ravel()

    best_by_key = {}
    for j, score in zip(compatible, sims):
        key = ref_keys[j]
        if score > best_by_key.get(key, -1):
            best_by_key[key] = float(score)

    ranked_keys = [
        key for key, score in sorted(
            best_by_key.items(),
            key=lambda item: item[1],
            reverse=True,
        )[:25]
    ]

    # Since the true structure is excluded, its rank is necessarily
    # outside this reference catalog. Keep this diagnostic explicit.
    rr_fair.append(0.0)

print("Queries evaluated:", len(queries))
print("Target-in-reference candidates:", 0)
print("MRR@25:", f"{np.mean(rr_fair):.5f}")


In [ ]:
# 1. Confirm query labels are present in the reference catalog.
ref_label_set = set(refs_all["inchikey14"].astype(str))
query_labels = queries["inchikey14"].astype(str)

print("Queries:", len(queries))
print("Query labels found in reference catalog:",
      query_labels.isin(ref_label_set).sum())

# 2. Check whether the query and reference labels have matching formats.
print("\nQuery label examples:")
print(query_labels.head().tolist())

print("\nReference label examples:")
print(refs_all["inchikey14"].astype(str).head().tolist())

# 3. Check ionization-mode compatibility for the true structures.
ref_mode_by_label = (
    refs_all.assign(
        inchikey14=refs_all["inchikey14"].astype(str),
        ionization_mode=refs_all["ionization_mode"].astype(str),
    )
    .groupby("inchikey14")["ionization_mode"]
    .agg(set)
    .to_dict()
)

mode_matches = sum(
    str(mode) in ref_mode_by_label.get(str(label), set())
    for label, mode in zip(
        queries["inchikey14"], queries["ionization_mode"]
    )
)

print("\nTargets with a reference in the same ionization mode:", mode_matches)

In [ ]:
# Inspect the first 10 queries against their true reference structures.
for i in range(10):
    true_label = str(queries.iloc[i]["inchikey14"])
    query_mode = str(queries.iloc[i]["ionization_mode"])

    # Reference rows for this exact structure and ionization mode
    mask = (
        (refs_all["inchikey14"].astype(str) == true_label)
        & (refs_all["ionization_mode"].astype(str) == query_mode)
    )
    ref_idx = np.flatnonzero(mask.to_numpy())

    if len(ref_idx) == 0:
        print(i, "No matching-mode reference")
        continue

    # Similarity between this query and its true reference spectrum
    sim = float((Q[i] @ R[ref_idx].T).toarray().max())

    print(
        i,
        "true label:", true_label,
        "| mode:", query_mode,
        "| query features:", Q[i].nnz,
        "| reference features:", R[ref_idx].nnz,
        "| true similarity:", round(sim, 6),
    )

In [ ]:
# Recompute ranks for a small sample, explicitly checking sparse indices.
for i in range(10):
    true_label = str(queries.iloc[i]["inchikey14"])
    query_mode = str(queries.iloc[i]["ionization_mode"])

    compatible_idx = np.flatnonzero(ref_modes == query_mode)
    sims = (Q[i] @ R[compatible_idx].T).tocsr()

    # Candidate positions returned by the sparse multiplication
    positions = sims.indices
    scores = sims.data
    labels = ref_labels[compatible_idx[positions]]

    if len(scores) == 0:
        print(i, "No positive similarities")
        continue

    order = np.argsort(-scores, kind="stable")
    labels_ranked = labels[order]
    scores_ranked = scores[order]

    hits = np.flatnonzero(labels_ranked == true_label)

    if len(hits):
        rank = int(hits[0]) + 1
        print(
            i,
            "true similarity found:",
            round(float(scores_ranked[hits[0]]), 6),
            "| rank:", rank,
        )
    else:
        print(
            i,
            "target not among positive-score candidates",
            "| positive candidates:", len(scores),
        )

In [ ]:
reciprocal_ranks = []
hits_at_25 = []
rank_counts = {}

for start in range(0, len(queries), 32):
    end = min(start + 32, len(queries))
    similarities = Q[start:end] @ R.T

    for local_i, global_i in enumerate(range(start, end)):
        true_label = str(queries.iloc[global_i]["inchikey14"])
        query_mode = str(queries.iloc[global_i]["ionization_mode"])

        # Keep only candidates in the same ionization mode.
        compatible_idx = np.flatnonzero(ref_modes == query_mode)
        row = (Q[global_i] @ R[compatible_idx].T).tocsr()

        positions = row.indices
        scores = row.data
        labels = ref_labels[compatible_idx[positions]]

        order = np.argsort(-scores, kind="stable")
        ranked_labels = labels[order]
        matches = np.flatnonzero(ranked_labels == true_label)

        if len(matches) and matches[0] < 25:
            rank = int(matches[0]) + 1
            reciprocal_ranks.append(1.0 / rank)
            hits_at_25.append(1)
            rank_counts[rank] = rank_counts.get(rank, 0) + 1
        else:
            reciprocal_ranks.append(0.0)
            hits_at_25.append(0)

print("Queries evaluated:", len(queries))
print(f"MRR@25: {np.mean(reciprocal_ranks):.5f}")
print(f"Target in top 25: {np.mean(hits_at_25):.1%}")
print("Exact rank-1 hits:", rank_counts.get(1, 0))

In [ ]:
reciprocal_ranks = []
hits_at_25 = []
rank_counts = {}

for start in range(0, len(queries), 32):
    end = min(start + 32, len(queries))
    similarities = Q[start:end] @ R.T

    for local_i, global_i in enumerate(range(start, end)):
        true_label = str(queries.iloc[global_i]["inchikey14"])
        query_mode = str(queries.iloc[global_i]["ionization_mode"])

        # Keep only candidates in the same ionization mode.
        compatible_idx = np.flatnonzero(ref_modes == query_mode)
        row = (Q[global_i] @ R[compatible_idx].T).tocsr()

        positions = row.indices
        scores = row.data
        labels = ref_labels[compatible_idx[positions]]

        order = np.argsort(-scores, kind="stable")
        ranked_labels = labels[order]
        matches = np.flatnonzero(ranked_labels == true_label)

        if len(matches) and matches[0] < 25:
            rank = int(matches[0]) + 1
            reciprocal_ranks.append(1.0 / rank)
            hits_at_25.append(1)
            rank_counts[rank] = rank_counts.get(rank, 0) + 1
        else:
            reciprocal_ranks.append(0.0)
            hits_at_25.append(0)

print("Queries evaluated:", len(queries))
print(f"MRR@25: {np.mean(reciprocal_ranks):.5f}")
print(f"Target in top 25: {np.mean(hits_at_25):.1%}")
print("Exact rank-1 hits:", rank_counts.get(1, 0))

In [ ]:
# Rebuild one reference spectrum per structure
refs_all = (
    reference_spectra
    .groupby("inchikey14", sort=False)
    .head(1)
    .reset_index(drop=True)
)

print("Reference spectra:", len(refs_all))
print("Reference structures:", refs_all["inchikey14"].nunique())

In [ ]:
required = ["queries", "refs_all", "Q", "R", "np"]

for name in required:
    print(f"{name}: {'OK' if name in globals() else 'MISSING'}")

In [ ]:
print("refs_all rows:", len(refs_all))
print("R shape:", R.shape)
print("Q shape:", Q.shape)
print("Reference labels:", len(ref_labels))

In [ ]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.preprocessing import normalize
import numpy as np

# Recreate the reference set
refs_all = (
    reference_spectra
    .groupby("inchikey14", sort=False)
    .head(1)
    .reset_index(drop=True)
)

# Recreate features in exactly this order
ref_dicts = [
    spectrum_to_dict(mz, intensity)
    for mz, intensity in zip(
        refs_all["ms2_mzs"],
        refs_all["ms2_normalized_intensities"],
    )
]

query_dicts = [
    spectrum_to_dict(mz, intensity)
    for mz, intensity in zip(
        queries["ms2_mzs"],
        queries["ms2_normalized_intensities"],
    )
]

vectorizer = DictVectorizer(dtype=np.float32)
R = normalize(vectorizer.fit_transform(ref_dicts), norm="l2")
Q = normalize(vectorizer.transform(query_dicts), norm="l2")

ref_labels = refs_all["inchikey14"].astype(str).to_numpy()
ref_modes = refs_all["ionization_mode"].astype(str).to_numpy()

print("Reference rows:", len(refs_all))
print("R rows:", R.shape[0])
print("Query rows:", Q.shape[0])

In [96]:
print("reference_spectra:", len(reference_spectra))
print("refs_all:", len(refs_all))
print("R shape:", R.shape)
print("Q shape:", Q.shape)
print("queries:", len(queries))

reference_spectra: 2156469
refs_all: 275810
R shape: (275810, 47501)
Q shape: (2000, 47501)
queries: 2000


In [97]:
import numpy as np
from sklearn.feature_extraction import DictVectorizer
from sklearn.preprocessing import normalize

# Rebuild reference spectra and feature matrix as one consistent unit
refs_all = (
    reference_spectra
    .groupby("inchikey14", sort=False)
    .head(1)
    .reset_index(drop=True)
)

ref_dicts = [
    spectrum_to_dict(mz, intensity)
    for mz, intensity in zip(
        refs_all["ms2_mzs"],
        refs_all["ms2_normalized_intensities"],
    )
]

R = normalize(
    vectorizer.transform(ref_dicts),
    norm="l2",
)

ref_labels = refs_all["inchikey14"].astype(str).to_numpy()
ref_modes = refs_all["ionization_mode"].astype(str).to_numpy()

assert R.shape[0] == len(refs_all) == len(ref_labels) == len(ref_modes)

print("Reference rows:", len(refs_all))
print("Matrix shape:", R.shape)
print("Reference labels:", len(ref_labels))

Reference rows: 275810
Matrix shape: (275810, 47501)
Reference labels: 275810


In [99]:
i = 4
true_label = str(queries.iloc[i]["inchikey14"])
query_mode = str(queries.iloc[i]["ionization_mode"])

mask = (
    (ref_labels == true_label)
    & (ref_modes == query_mode)
)
ref_idx = np.flatnonzero(mask)

if len(ref_idx):
    sim = float((Q[i] @ R[ref_idx].T).toarray().max())
    print("True-structure similarity:", round(sim, 6))
else:
    print("No reference for this structure in the same ionization mode.")

True-structure similarity: 0.56379


In [100]:
i = 4

true_label = str(queries.iloc[i]["inchikey14"])
query_mode = str(queries.iloc[i]["ionization_mode"])

mask = (
    (ref_labels == true_label)
    & (ref_modes == query_mode)
)
ref_idx = np.flatnonzero(mask)

if len(ref_idx):
    sim = float((Q[i] @ R[ref_idx].T).toarray().max())
    print("True-structure similarity:", round(sim, 6))
else:
    print("No matching-mode reference found.")

True-structure similarity: 0.56379


In [102]:
# Step 3A: Verify the current benchmark state

print("Queries:", len(queries))
print("Reference structures:", len(refs_all))
print("Query feature matrix:", Q.shape)
print("Reference feature matrix:", R.shape)

assert Q.shape[0] == len(queries)
assert R.shape[0] == len(refs_all)
assert R.shape[0] == len(ref_labels)
assert R.shape[0] == len(ref_modes)

print("\nPASS: Matrix dimensions and reference labels are aligned.")

Queries: 2000
Reference structures: 275810
Query feature matrix: (2000, 47501)
Reference feature matrix: (275810, 47501)

PASS: Matrix dimensions and reference labels are aligned.


In [103]:
# Step 3B: Inspect the adducts we need to support

print("Test-set adduct counts:")
print(test["adduct"].value_counts(dropna=False))

print("\nTraining-set adduct counts:")
print(train["adduct"].value_counts().head(12))

Test-set adduct counts:
adduct
[M+H]+          959
[M-H]-          193
[M+CH2O2-H]-     31
[M+Na]+          22
[M+NH4]+          4
[M+K]+            2
[M+Cl]-           2
Name: count, dtype: int64

Training-set adduct counts:


KeyError: 'adduct'

In [104]:
print("Training variable type:", type(train))
print("\nAvailable columns:")
print(train.columns.tolist())

Training variable type: <class 'pandas.core.frame.DataFrame'>

Available columns:
['normalized_smiles', 'inchikey14', 'molecular_formula']


In [105]:
for name, obj in list(globals().items()):
    if isinstance(obj, __import__("pandas").DataFrame):
        print(name, obj.shape, list(obj.columns[:8]))

test (1213, 12) ['molecule_id', 'spectrum_id', 'ms2_mzs', 'ms2_normalized_intensities', 'base_peak_intensity', 'adduct', 'ionization_mode', 'instrument_type']
sample (400, 2) ['molecule_id', 'smiles']
chunk (131072, 5) ['precursor_mz', 'adduct', 'normalized_smiles', 'inchikey14', 'ingest_lib']
train_small (50000, 5) ['precursor_mz', 'adduct', 'normalized_smiles', 'inchikey14', 'ingest_lib']
group (446, 3) ['precursor_mz', 'adduct', 'ingest_lib']
submission (400, 2) ['molecule_id', 'smiles']
sub (400, 2) ['molecule_id', 'smiles']
df (39608, 8) ['precursor_mz', 'adduct', 'ingest_lib', 'normalized_smiles', 'molecular_formula', 'inchikey14', 'precursor_error_ppm', 'num_peaks']
mass_df (2539608, 3) ['precursor_mz', 'adduct', 'ingest_lib']
outliers (77, 8) ['precursor_mz', 'adduct', 'ingest_lib', 'normalized_smiles', 'molecular_formula', 'inchikey14', 'precursor_error_ppm', 'num_peaks']
unusual (0, 8) ['precursor_mz', 'adduct', 'ingest_lib', 'normalized_smiles', 'molecular_formula', 'inchike

In [106]:
print("Training adduct counts:")
print(train_meta["adduct"].value_counts().head(12))

print("\nTest adduct counts:")
print(test["adduct"].value_counts())

print("\nTraining ionization modes:")
print(train_meta["ionization_mode"].value_counts())

Training adduct counts:
adduct
[M+H]+          1324585
[M-H]-           465688
[2M+Na]+         190033
[M+Na]+          178092
[2M+H]+           92041
[M+CH2O2-H]-      83188
[2M-H]-           56172
[M+NH4]+          53452
[M+Cl]-           27511
[M-H2O+H]+        21682
[M+K]+            18955
[M]+               8818
Name: count, dtype: int64

Test adduct counts:
adduct
[M+H]+          959
[M-H]-          193
[M+CH2O2-H]-     31
[M+Na]+          22
[M+NH4]+          4
[M+K]+            2
[M+Cl]-           2
Name: count, dtype: int64

Training ionization modes:
ionization_mode
positive    1899640
negative     639968
Name: count, dtype: int64


In [107]:
# Step 3D: Validate adduct offsets against training records

import numpy as np
import pandas as pd

adduct_offsets = {
    "[M+H]+": 1.007276,
    "[M-H]-": -1.007276,
    "[M+Na]+": 22.989218,
    "[M+NH4]+": 18.033823,
    "[M+K]+": 38.963158,
    "[M+Cl]-": 34.969402,
    "[M+CH2O2-H]-": 44.998201,
}

# Join a sample of training records to their RDKit exact masses.
mass_check = (
    train_meta[["inchikey14", "precursor_mz", "adduct"]]
    .merge(
        library[["inchikey14", "exact_mass"]],
        on="inchikey14",
        how="inner",
    )
)

mass_check = mass_check[
    mass_check["adduct"].isin(adduct_offsets)
].copy()

mass_check["offset"] = mass_check["adduct"].map(adduct_offsets)
mass_check["predicted_mz"] = (
    mass_check["exact_mass"] + mass_check["offset"]
)
mass_check["error_ppm_calc"] = (
    (mass_check["precursor_mz"] - mass_check["predicted_mz"])
    / mass_check["predicted_mz"]
    * 1e6
)

print(
    mass_check.groupby("adduct")["error_ppm_calc"]
    .agg(["count", "median", "mean"])
    .round(3)
)

                count  median       mean
adduct                                  
[M+CH2O2-H]-    83188   0.022      1.567
[M+Cl]-         27511   0.002    204.263
[M+H]+        1324585   0.821    821.897
[M+K]+          18955  -0.009   4083.088
[M+NH4]+        53452   0.020   1597.860
[M+Na]+        178092  -0.027   2246.853
[M-H]-         465688   0.069  22311.385


In [108]:
error_summary = (
    mass_check.groupby("adduct")["error_ppm_calc"]
    .quantile([0.50, 0.90, 0.95, 0.99])
    .unstack()
)
error_summary.columns = ["p50", "p90", "p95", "p99"]

error_summary["within_10_ppm"] = (
    mass_check.assign(
        within_10=mass_check["error_ppm_calc"].abs() <= 10
    )
    .groupby("adduct")["within_10"]
    .mean() * 100
)

print(error_summary.round(2))

               p50   p90    p95        p99  within_10_ppm
adduct                                                   
[M+CH2O2-H]-  0.02  0.77   1.22       2.22          99.92
[M+Cl]-       0.00  0.12   0.41       1.57          99.85
[M+H]+        0.82  3.17   3.80      45.06          95.19
[M+K]+       -0.01  1.26   2.56  160660.14          94.15
[M+NH4]+      0.02  1.17   2.26      20.15          96.46
[M+Na]+      -0.03  7.69  15.12   89010.12          80.85
[M-H]-        0.07  1.61   2.12     667.36          96.45


In [109]:
# Count how many validation spectra have a matching structure
# within ±10 ppm, using the expected adduct mass offset.

val_mass = val_spectra[
    ["inchikey14", "precursor_mz", "adduct"]
].merge(
    library[["inchikey14", "exact_mass"]],
    on="inchikey14",
    how="left"
)

val_mass["offset"] = val_mass["adduct"].map(adduct_offsets)
val_mass["predicted_mz"] = val_mass["exact_mass"] + val_mass["offset"]

val_mass["error_ppm"] = (
    (val_mass["precursor_mz"] - val_mass["predicted_mz"])
    / val_mass["predicted_mz"] * 1e6
)

val_mass["within_10_ppm"] = val_mass["error_ppm"].abs() <= 10

print("Validation spectra:", len(val_mass))
print(
    "Known structure within ±10 ppm:",
    f"{val_mass['within_10_ppm'].mean() * 100:.2f}%"
)
print("\nBy adduct:")
print(
    val_mass.groupby("adduct")["within_10_ppm"]
    .agg(["count", "mean"])
    .assign(percent=lambda x: x["mean"] * 100)
    .drop(columns="mean")
    .round(2)
)

Validation spectra: 254378
Known structure within ±10 ppm: 79.91%

By adduct:
                 count  percent
adduct                         
[2M+C2H4O2-H]-      15     0.00
[2M+CH2O2-H]-       66     0.00
[2M+Ca]2+            1     0.00
[2M+H]+           9231     0.00
[2M+K]+              6     0.00
[2M+NH4]+           49     0.00
[2M+Na-2H]-        116     0.00
[2M+Na2-H]+          1     0.00
[2M+Na]+         19102     0.00
[2M-H3O2]+           2     0.00
[2M-H5O3]+           1     0.00
[2M-HO]+             2     0.00
[2M-H]-           5758     0.00
[M+2H]2+           176     0.00
[M+2Na-H]+          16     0.00
[M+3H]3+            18     0.00
[M+Br]-              3     0.00
[M+C2F3O2]-          2     0.00
[M+C2H2NaO2]-        1     0.00
[M+C2H3NNa]+         3     0.00
[M+C2H4N]+          23     0.00
[M+C2H4O2-H]-      680     0.00
[M+C2H5N]2+          1     0.00
[M+CH2O2-H]-      8213    99.91
[M+CH5O]+            4     0.00
[M+Ca-H]+            2     0.00
[M+Ca]2+             6    

In [110]:
unsupported = (
    val_spectra.loc[
        ~val_spectra["adduct"].isin(adduct_offsets),
        "adduct"
    ]
    .value_counts()
)

print("Validation spectra with unsupported adducts:", unsupported.sum())
print("\nMost common unsupported adducts:")
print(unsupported.head(20))

Validation spectra with unsupported adducts: 38819

Most common unsupported adducts:
adduct
[2M+Na]+         19102
[2M+H]+           9231
[2M-H]-           5758
[M-H2O+H]+        1998
[M]+               883
[M+C2H4O2-H]-      680
[M-2H2O+H]+        326
[M+2H]2+           176
[2M+Na-2H]-        116
[M]-                67
[2M+CH2O2-H]-       66
[M-CH3]-            63
[M-2H]-             54
[2M+NH4]+           49
[M-H2O]+            45
[M-H5O3]+           26
[M+C2H4N]+          23
[M+3H]3+            18
[M-H2O-H]-          17
[M+2Na-H]+          16
Name: count, dtype: int64


In [111]:
import re

PROTON = 1.0072764666
SODIUM = 22.989218
POTASSIUM = 38.963158
AMMONIUM = 18.033823
CHLORIDE = 34.969402
FORMIC_ACID_MINUS_H = 44.998201
WATER = 18.010565

def expected_mz(neutral_mass, adduct):
    # Dimer ions
    if adduct == "[2M+H]+":
        return (2 * neutral_mass + PROTON)
    if adduct == "[2M+Na]+":
        return (2 * neutral_mass + SODIUM)
    if adduct == "[2M+NH4]+":
        return (2 * neutral_mass + AMMONIUM)
    if adduct == "[2M-H]-":
        return (2 * neutral_mass - PROTON)
    if adduct == "[2M+CH2O2-H]-":
        return (2 * neutral_mass + FORMIC_ACID_MINUS_H)

    # Common singly charged ions
    offsets = {
        "[M+H]+": PROTON,
        "[M-H]-": -PROTON,
        "[M+Na]+": SODIUM,
        "[M+K]+": POTASSIUM,
        "[M+NH4]+": AMMONIUM,
        "[M+Cl]-": CHLORIDE,
        "[M+CH2O2-H]-": FORMIC_ACID_MINUS_H,
        "[M]+": 0.0,
        "[M]-": 0.0,
        "[M-H2O+H]+": PROTON - WATER,
        "[M-2H2O+H]+": PROTON - 2 * WATER,
        "[M-H2O]+": -WATER,
        "[M-H2O-H]-": -WATER - PROTON,
        "[M-2H]-": -2 * PROTON,
        "[M-CH3]-": -15.023475,
    }
    if adduct in offsets:
        return neutral_mass + offsets[adduct]

    # Multiply charged ions
    if adduct == "[M+2H]2+":
        return (neutral_mass + 2 * PROTON) / 2
    if adduct == "[M+3H]3+":
        return (neutral_mass + 3 * PROTON) / 3

    return None

print("Calculator defined. Next, we'll test its coverage on validation spectra.")

Calculator defined. Next, we'll test its coverage on validation spectra.


In [112]:
# Calculate expected precursor m/z for each validation spectrum.
val_mass = val_spectra[
    ["inchikey14", "precursor_mz", "adduct"]
].merge(
    library[["inchikey14", "exact_mass"]],
    on="inchikey14",
    how="left"
)

val_mass["predicted_mz"] = [
    expected_mz(mass, adduct)
    for mass, adduct in zip(
        val_mass["exact_mass"],
        val_mass["adduct"]
    )
]

val_mass["error_ppm"] = (
    (val_mass["precursor_mz"] - val_mass["predicted_mz"])
    / val_mass["predicted_mz"] * 1e6
)

supported = (
    val_mass["predicted_mz"].notna()
    & np.isfinite(val_mass["predicted_mz"])
    & (val_mass["predicted_mz"] > 0)
)

print(f"Validation spectra: {len(val_mass):,}")
print(f"Supported adducts: {supported.sum():,} ({supported.mean()*100:.2f}%)")
print(f"Unsupported adducts: {(~supported).sum():,}")

print("\nError percentiles for supported adducts:")
print(
    val_mass.loc[supported, "error_ppm"]
    .abs()
    .quantile([0.50, 0.90, 0.95, 0.99])
    .rename(index={
        0.50: "median absolute ppm",
        0.90: "90th percentile",
        0.95: "95th percentile",
        0.99: "99th percentile",
    })
    .round(2)
)

print("\nFraction within tolerance:")
for ppm in [5, 10, 20, 50]:
    fraction = (
        val_mass.loc[supported, "error_ppm"].abs() <= ppm
    ).mean() * 100
    print(f"±{ppm:>2} ppm: {fraction:.2f}%")

Validation spectra: 254,378
Supported adducts: 253,412 (99.62%)
Unsupported adducts: 966

Error percentiles for supported adducts:
median absolute ppm       0.89
90th percentile           3.69
95th percentile          10.11
99th percentile        2062.47
Name: error_ppm, dtype: float64

Fraction within tolerance:
± 5 ppm: 92.64%
±10 ppm: 94.91%
±20 ppm: 96.87%
±50 ppm: 97.82%


In [113]:
# Summarize mass errors by adduct for supported validation spectra.
val_mass["abs_error_ppm"] = val_mass["error_ppm"].abs()

supported_rows = val_mass[
    val_mass["predicted_mz"].notna()
    & np.isfinite(val_mass["error_ppm"])
    & (val_mass["predicted_mz"] > 0)
].copy()

adduct_error_summary = (
    supported_rows.groupby("adduct")
    .agg(
        count=("error_ppm", "size"),
        median_abs_ppm=("abs_error_ppm", "median"),
        p95_abs_ppm=("abs_error_ppm", lambda x: x.quantile(0.95)),
        pct_within_10ppm=(
            "abs_error_ppm",
            lambda x: (x <= 10).mean() * 100
        ),
        pct_over_100ppm=(
            "abs_error_ppm",
            lambda x: (x > 100).mean() * 100
        ),
    )
    .sort_values("count", ascending=False)
)

print(adduct_error_summary.round(2).to_string())

                count  median_abs_ppm  p95_abs_ppm  pct_within_10ppm  pct_over_100ppm
adduct                                                                               
[M+H]+         132032            1.39        10.10             95.00             1.90
[M-H]-          47046            0.23         3.30             96.24             2.74
[2M+Na]+        19102            1.14         3.57             99.99             0.00
[M+Na]+         17669            2.31        27.77             79.90             2.74
[2M+H]+          9231            1.24         3.95             99.94             0.02
[M+CH2O2-H]-     8213            0.08         1.41             99.91             0.02
[M+NH4]+         5859            0.12         4.11             96.59             3.38
[2M-H]-          5758            0.79         3.26             99.95             0.05
[M+Cl]-          2757            0.08         1.38             99.89             0.11
[M-H2O+H]+       1998            0.19      1861.62    

In [115]:
# Summarize the error distribution by structure and adduct.
outlier_summary = (
    supported_rows
    .groupby(["inchikey14", "adduct"])
    .agg(
        rows=("error_ppm", "size"),
        median_abs_ppm=("abs_error_ppm", "median"),
        max_abs_ppm=("abs_error_ppm", "max"),
        observed_mz_min=("precursor_mz", "min"),
        observed_mz_max=("precursor_mz", "max"),
        predicted_mz=("predicted_mz", "median"),
    )
    .reset_index()
)

outlier_summary = outlier_summary[
    outlier_summary["max_abs_ppm"] > 100
].sort_values("max_abs_ppm", ascending=False)

print("Structure/adduct groups with errors >100 ppm:",
      len(outlier_summary))

print("\nLargest 20 groups:")
print(outlier_summary.head(20).to_string(index=False))

Structure/adduct groups with errors >100 ppm: 974

Largest 20 groups:
    inchikey14 adduct  rows  median_abs_ppm  max_abs_ppm  observed_mz_min  observed_mz_max  predicted_mz
IGUZJYCAXLYZEE [M-H]-     4    2.629907e+06 2.629908e+06         482.9614         482.9615    133.050632
KJLXVAPBVMFHOL [M-H]-     1    1.314075e+06 1.314075e+06         266.0860         266.0860    114.985924
KJLXVAPBVMFHOL [M+H]+     1    1.291444e+06 1.291444e+06         268.1000         268.1000    117.000477
FLTWXIGFXYFAFG [M+H]+     1    1.282293e+06 1.282293e+06         527.2700         527.2700    231.026430
GTZCVFVGUGFEME [M+H]+    39    7.786695e+02 1.108343e+06         129.0200         369.0100    175.023714
FASDKYOPVNHBLU [M+H]+    86    2.352141e-02 1.098509e+06         212.0820         445.1390    212.121595
FFGPTBGBLSHEPO [M+H]+   197    1.664708e-01 1.088470e+06         237.1000         495.1810    237.102239
GNXFOGHNGIVQEH [M+H]+    90    3.938005e-03 1.086639e+06         242.1020         505.1800

In [114]:
# Inspect the largest mass errors without changing any data.
outlier_cols = [
    "inchikey14",
    "adduct",
    "precursor_mz",
    "exact_mass",
    "predicted_mz",
    "error_ppm",
]

worst_errors = (
    supported_rows.loc[
        supported_rows["abs_error_ppm"] > 100,
        outlier_cols
    ]
    .assign(
        abs_error_ppm=lambda x: x["error_ppm"].abs()
    )
    .sort_values("abs_error_ppm", ascending=False)
)

print("Rows with absolute error >100 ppm:", len(worst_errors))
print("\nCounts by adduct:")
print(worst_errors["adduct"].value_counts().head(15))

print("\nLargest 15 errors:")
print(worst_errors.head(15).to_string(index=False))

Rows with absolute error >100 ppm: 5188

Counts by adduct:
adduct
[M+H]+          2509
[M-H]-          1291
[M+Na]+          484
[M]+             231
[M+NH4]+         198
[M-H2O+H]+       124
[M+K]+           123
[M-2H2O+H]+       67
[M-CH3]-          56
[M-2H]-           52
[M+2H]2+          25
[M]-              18
[2M-H]-            3
[M+Cl]-            3
[M+CH2O2-H]-       2
Name: count, dtype: int64

Largest 15 errors:
    inchikey14 adduct  precursor_mz  exact_mass  predicted_mz    error_ppm  abs_error_ppm
IGUZJYCAXLYZEE [M-H]-      482.9615  134.057909    133.050632 2.629908e+06   2.629908e+06
IGUZJYCAXLYZEE [M-H]-      482.9615  134.057909    133.050632 2.629908e+06   2.629908e+06
IGUZJYCAXLYZEE [M-H]-      482.9614  134.057909    133.050632 2.629907e+06   2.629907e+06
IGUZJYCAXLYZEE [M-H]-      482.9614  134.057909    133.050632 2.629907e+06   2.629907e+06
KJLXVAPBVMFHOL [M-H]-      266.0860  115.993200    114.985924 1.314075e+06   1.314075e+06
KJLXVAPBVMFHOL [M+H]+      268.10

In [116]:
# Inspect individual validation rows for one suspicious structure.
example_key = "FASDKYOPVNHBLU"

example_rows = (
    supported_rows.loc[
        supported_rows["inchikey14"] == example_key,
        [
            "inchikey14",
            "adduct",
            "precursor_mz",
            "exact_mass",
            "predicted_mz",
            "error_ppm",
        ],
    ]
    .copy()
)

example_rows["abs_error_ppm"] = example_rows["error_ppm"].abs()
example_rows = example_rows.sort_values("abs_error_ppm", ascending=False)

print("Rows for structure:", len(example_rows))
print("\nLargest errors:")
print(example_rows.head(10).to_string(index=False))

print("\nSmallest errors:")
print(
    example_rows.sort_values("abs_error_ppm")
    .head(10)
    .to_string(index=False)
)

Rows for structure: 86

Largest errors:
    inchikey14 adduct  precursor_mz  exact_mass  predicted_mz     error_ppm  abs_error_ppm
FASDKYOPVNHBLU [M+H]+       445.139  211.114319    212.121595  1.098509e+06   1.098509e+06
FASDKYOPVNHBLU [M+H]+       445.139  211.114319    212.121595  1.098509e+06   1.098509e+06
FASDKYOPVNHBLU [M+H]+       445.139  211.114319    212.121595  1.098509e+06   1.098509e+06
FASDKYOPVNHBLU [M+H]+       423.158  211.114319    212.121595  9.948841e+05   9.948841e+05
FASDKYOPVNHBLU [M+H]+       423.158  211.114319    212.121595  9.948841e+05   9.948841e+05
FASDKYOPVNHBLU [M+H]+       423.158  211.114319    212.121595  9.948841e+05   9.948841e+05
FASDKYOPVNHBLU [M+H]+       234.064  211.114319    212.121595  1.034426e+05   1.034426e+05
FASDKYOPVNHBLU [M+H]+       234.064  211.114319    212.121595  1.034426e+05   1.034426e+05
FASDKYOPVNHBLU [M+H]+       234.064  211.114319    212.121595  1.034426e+05   1.034426e+05
FASDKYOPVNHBLU [M+H]+       212.082  211.114319   

In [117]:
# Categorize mass errors for each supported adduct.
supported_rows["error_category"] = pd.cut(
    supported_rows["abs_error_ppm"],
    bins=[-1, 10, 50, 100, float("inf")],
    labels=["<=10 ppm", "10-50 ppm", "50-100 ppm", ">100 ppm"]
)

error_counts = pd.crosstab(
    supported_rows["adduct"],
    supported_rows["error_category"],
    normalize="index"
) * 100

# Show the most frequent adducts first.
adduct_order = supported_rows["adduct"].value_counts().index
error_counts = error_counts.reindex(adduct_order)

print(error_counts.head(15).round(2).to_string())

error_category  <=10 ppm  10-50 ppm  50-100 ppm  >100 ppm
adduct                                                   
[M+H]+             95.00       3.02        0.09      1.90
[M-H]-             96.24       0.58        0.43      2.74
[2M+Na]+           99.99       0.01        0.00      0.00
[M+Na]+            79.90      17.24        0.12      2.74
[2M+H]+            99.94       0.04        0.00      0.02
[M+CH2O2-H]-       99.91       0.06        0.00      0.02
[M+NH4]+           96.59       0.03        0.00      3.38
[2M-H]-            99.95       0.00        0.00      0.05
[M+Cl]-            99.89       0.00        0.00      0.11
[M-H2O+H]+         92.24       1.35        0.20      6.21
[M+K]+             93.09       0.61        0.10      6.20
[M]+               73.39       0.34        0.11     26.16
[M-2H2O+H]+        79.45       0.00        0.00     20.55
[M+2H]2+           82.95       2.84        0.00     14.20
[M]-               71.64       1.49        0.00     26.87


In [118]:
# Evaluate whether the known structure passes a mass filter.
# This tests mass coverage, not final ranking accuracy.

rng = np.random.default_rng(42)

mass_test = val_spectra.sample(
    n=min(500, len(val_spectra)),
    random_state=42
).copy()

mass_test = mass_test.merge(
    library[["inchikey14", "exact_mass"]],
    on="inchikey14",
    how="left"
)

mass_test["predicted_mz"] = [
    expected_mz(mass, adduct)
    for mass, adduct in zip(
        mass_test["exact_mass"],
        mass_test["adduct"]
    )
]

mass_test["abs_error_ppm"] = (
    (
        mass_test["precursor_mz"] - mass_test["predicted_mz"]
    )
    / mass_test["predicted_mz"]
    * 1e6
).abs()

valid_mass = (
    mass_test["predicted_mz"].notna()
    & np.isfinite(mass_test["predicted_mz"])
    & (mass_test["predicted_mz"] > 0)
    & np.isfinite(mass_test["abs_error_ppm"])
)

print("Sampled spectra:", len(mass_test))
print("Valid mass calculations:", int(valid_mass.sum()))

for ppm in [5, 10, 20, 50, 100]:
    passed = valid_mass & (mass_test["abs_error_ppm"] <= ppm)
    print(f"Within ±{ppm:>3} ppm: {passed.mean() * 100:.2f}%")

Sampled spectra: 500
Valid mass calculations: 500
Within ±  5 ppm: 95.20%
Within ± 10 ppm: 96.00%
Within ± 20 ppm: 98.00%
Within ± 50 ppm: 99.00%
Within ±100 ppm: 99.00%


In [119]:
# Count candidate structures within each precursor-mass tolerance.
# Use chunks to avoid building a huge spectra-by-library matrix.

mass_values = library["exact_mass"].to_numpy()
library_labels = library["inchikey14"].to_numpy()

candidate_counts = {ppm: [] for ppm in [5, 10, 20, 50]}

for row in mass_test.itertuples(index=False):
    observed_mz = row.precursor_mz
    adduct = row.adduct

    # Invert the adduct equation to estimate neutral mass.
    # expected_mz(neutral_mass, adduct) is the forward equation.
    def neutral_mass_from_mz(mz, adduct):
        if adduct in ["[2M+H]+", "[2M-H]-"]:
            offset = PROTON if adduct == "[2M+H]+" else -PROTON
            return (mz - offset) / 2
        if adduct == "[2M+Na]+":
            return (mz - SODIUM) / 2
        if adduct == "[2M+NH4]+":
            return (mz - AMMONIUM) / 2
        if adduct == "[2M+CH2O2-H]-":
            return (mz - FORMIC_ACID_MINUS_H) / 2
        if adduct == "[M+2H]2+":
            return mz * 2 - 2 * PROTON
        if adduct == "[M+3H]3+":
            return mz * 3 - 3 * PROTON

        offsets = {
            "[M+H]+": PROTON,
            "[M-H]-": -PROTON,
            "[M+Na]+": SODIUM,
            "[M+K]+": POTASSIUM,
            "[M+NH4]+": AMMONIUM,
            "[M+Cl]-": CHLORIDE,
            "[M+CH2O2-H]-": FORMIC_ACID_MINUS_H,
            "[M]+": 0.0,
            "[M]-": 0.0,
            "[M-H2O+H]+": PROTON - WATER,
            "[M-2H2O+H]+": PROTON - 2 * WATER,
            "[M-H2O]+": -WATER,
            "[M-H2O-H]-": -WATER - PROTON,
            "[M-2H]-": -2 * PROTON,
            "[M-CH3]-": -15.023475,
        }

        if adduct not in offsets:
            return np.nan
        return mz - offsets[adduct]

    neutral_mass = neutral_mass_from_mz(observed_mz, adduct)

    if not np.isfinite(neutral_mass) or neutral_mass <= 0:
        for ppm in candidate_counts:
            candidate_counts[ppm].append(0)
        continue

    # Approximate neutral-mass tolerance, in ppm.
    errors_ppm = np.abs(mass_values - neutral_mass) / neutral_mass * 1e6

    for ppm in candidate_counts:
        candidate_counts[ppm].append(int(np.sum(errors_ppm <= ppm)))

for ppm, counts in candidate_counts.items():
    counts = np.asarray(counts)
    print(
        f"±{ppm:>2} ppm | "
        f"median candidates: {np.median(counts):.0f} | "
        f"mean: {np.mean(counts):.1f} | "
        f"max: {np.max(counts)} | "
        f"zero candidates: {(counts == 0).sum()}"
    )

± 5 ppm | median candidates: 28 | mean: 56.7 | max: 423 | zero candidates: 12
±10 ppm | median candidates: 49 | mean: 76.2 | max: 429 | zero candidates: 9
±20 ppm | median candidates: 82 | mean: 113.2 | max: 536 | zero candidates: 4
±50 ppm | median candidates: 217 | mean: 278.0 | max: 978 | zero candidates: 0


In [120]:
# Measure true-structure retention using the same mass inversion
# as the candidate-count calculation.

retention_results = []

for i, row in enumerate(mass_test.itertuples(index=False)):
    neutral_mass = neutral_mass_from_mz(row.precursor_mz, row.adduct)

    result = {
        "inchikey14": row.inchikey14,
        "adduct": row.adduct,
        "neutral_mass": neutral_mass,
    }

    if not np.isfinite(neutral_mass) or neutral_mass <= 0:
        for ppm in [5, 10, 20, 50]:
            result[f"retained_{ppm}ppm"] = False
            result[f"count_{ppm}ppm"] = 0
        retention_results.append(result)
        continue

    errors_ppm = np.abs(mass_values - neutral_mass) / neutral_mass * 1e6

    true_mass_rows = library["inchikey14"].to_numpy() == row.inchikey14
    true_mass_errors = errors_ppm[true_mass_rows]

    for ppm in [5, 10, 20, 50]:
        result[f"retained_{ppm}ppm"] = bool(
            len(true_mass_errors) > 0 and np.min(true_mass_errors) <= ppm
        )
        result[f"count_{ppm}ppm"] = int(np.sum(errors_ppm <= ppm))

    retention_results.append(result)

retention_df = pd.DataFrame(retention_results)

for ppm in [5, 10, 20, 50]:
    print(
        f"±{ppm:>2} ppm | "
        f"true structure retained: "
        f"{retention_df[f'retained_{ppm}ppm'].mean() * 100:.2f}% | "
        f"median candidates: "
        f"{retention_df[f'count_{ppm}ppm'].median():.0f}"
    )

± 5 ppm | true structure retained: 95.20% | median candidates: 28
±10 ppm | true structure retained: 96.00% | median candidates: 49
±20 ppm | true structure retained: 98.00% | median candidates: 82
±50 ppm | true structure retained: 99.00% | median candidates: 217


In [121]:
print("Queries:", len(queries))
print("Reference structures:", len(refs_all))
print("Query matrix:", Q.shape)
print("Reference matrix:", R.shape)
print("Reference labels:", len(refs_all["inchikey14"]))

print("\nRequired columns:")
print("queries:", {"inchikey14", "precursor_mz", "adduct", "ionization_mode"}.issubset(queries.columns))
print("refs_all:", {"inchikey14", "precursor_mz", "adduct", "ionization_mode"}.issubset(refs_all.columns))

Queries: 2000
Reference structures: 275810
Query matrix: (2000, 47501)
Reference matrix: (275810, 47501)
Reference labels: 275810

Required columns:
queries: True
refs_all: True


In [122]:

import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity

# Confirm that reference labels and similarity-matrix rows align.
assert R.shape[0] == len(refs_all)
assert Q.shape[0] == len(queries)
assert len(refs_all["inchikey14"]) == R.shape[0]

query_labels = queries["inchikey14"].astype(str).to_numpy()
reference_labels = refs_all["inchikey14"].astype(str).to_numpy()

query_mz = queries["precursor_mz"].to_numpy(dtype=float)
reference_mz = refs_all["precursor_mz"].to_numpy(dtype=float)

print("Alignment checks passed.")
print("Queries:", len(query_labels))
print("Reference structures:", len(reference_labels))


Alignment checks passed.
Queries: 2000
Reference structures: 275810


In [123]:

from scipy.sparse import csr_matrix

# R and Q are already L2-normalized sparse fragment feature matrices.
# Compute cosine similarity for manageable query batches.
batch_size = 50
mass_tolerances = [10, 20, 50]
methods = ["mass_only", "fragments_only", "combined"]

results = []

for start in range(0, len(queries), batch_size):
    end = min(start + batch_size, len(queries))

    # Sparse matrix multiplication; each row is a query.
    similarity_batch = (Q[start:end] @ R.T).toarray()

    for local_i, q_idx in enumerate(range(start, end)):
        q_label = query_labels[q_idx]
        q_mz = query_mz[q_idx]

        # Symmetric ppm error based on query precursor m/z.
        ppm_error = np.abs(reference_mz - q_mz) / max(abs(q_mz), 1e-12) * 1e6
        sim = similarity_batch[local_i]

        # Evaluate only candidates within the chosen mass tolerance
        # for combined ranking; mass-only uses the same candidate sets.
        for tol in mass_tolerances:
            eligible = ppm_error <= tol

            for method in methods:
                if method == "fragments_only":
                    scores = sim.copy()
                elif method == "mass_only":
                    scores = -ppm_error
                else:
                    # Rank primarily by fragment similarity, with a small
                    # mass-error preference as a tie-breaker.
                    scores = sim - 0.001 * (ppm_error / tol)

                scores = scores.copy()
                scores[~eligible] = -np.inf

                top_idx = np.argsort(scores)[::-1][:25]
                top_labels = reference_labels[top_idx]
                hits = np.flatnonzero(top_labels == q_label)

                reciprocal_rank = 1.0 / (hits[0] + 1) if len(hits) else 0.0
                hit_at_25 = int(len(hits) > 0)

                results.append({
                    "query_idx": q_idx,
                    "tolerance_ppm": tol,
                    "method": method,
                    "rr_at_25": reciprocal_rank,
                    "hit_at_25": hit_at_25,
                    "candidate_count": int(eligible.sum()),
                })

    if (start // batch_size) % 5 == 0:
        print(f"Processed {end}/{len(queries)} queries")

benchmark = pd.DataFrame(results)

summary = (
    benchmark.groupby(["tolerance_ppm", "method"], as_index=False)
    .agg(
        MRR_at_25=("rr_at_25", "mean"),
        Hit_at_25=("hit_at_25", "mean"),
        Median_candidates=("candidate_count", "median"),
    )
)

display(summary.round(4))


Processed 50/2000 queries
Processed 300/2000 queries
Processed 550/2000 queries
Processed 800/2000 queries
Processed 1050/2000 queries
Processed 1300/2000 queries
Processed 1550/2000 queries
Processed 1800/2000 queries


,tolerance_ppm,method,MRR_at_25,Hit_at_25,Median_candidates
0,10,combined,0.3536,0.4220,21.0
1,10,fragments_only,0.3492,0.4220,21.0
2,10,mass_only,0.2781,0.4215,21.0
3,20,combined,0.3479,0.4230,36.0
4,20,fragments_only,0.3405,0.4180,36.0
5,20,mass_only,0.2855,0.4265,36.0
6,50,combined,0.3379,0.4165,93.0
7,50,fragments_only,0.3318,0.4105,93.0
8,50,mass_only,0.2822,0.4280,93.0


In [124]:

# Step 1: Check the columns needed for adduct-aware candidate retrieval.

print("Library shape:", library.shape)
print("Library columns:", library.columns.tolist())
print("Query columns:", queries.columns.tolist())

print("\nExample candidate structures:")
display(library.head(5))

print("\nAdduct counts in validation queries:")
display(queries["adduct"].value_counts().head(15))


Library shape: (275810, 5)
Library columns: ['normalized_smiles', 'inchikey14', 'molecular_formula', 'exact_mass', 'rdkit_formula']
Query columns: ['inchikey14', 'normalized_smiles', 'precursor_mz', 'adduct', 'ionization_mode', 'ms2_mzs', 'ms2_normalized_intensities']

Example candidate structures:


,normalized_smiles,inchikey14,molecular_formula,exact_mass,rdkit_formula
0,O=C1NC(=O)c2cc(Nc3ccccc3)c(Nc3ccccc3)cc21,AAALVYBICLMAMA,C20H15N3O2,329.116427,C20H15N3O2
1,C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1,AAKJLRGGTJKAMG,C22H23N3O4,393.168856,C22H23N3O4
2,CC1(C)CCC(C)(C)c2cc(C(O)C(O)=Nc3ccc(C(=O)O)cc3...,AANFHDFOMFRLLR,C23H26FNO4,399.184587,C23H26FNO4
3,CN1C(=O)CN=C(c2ccccc2)c2cc(Cl)ccc21,AAOVKJBEBIDNHE,C16H13ClN2O,284.071641,C16H13ClN2O
4,NC(=O)C1=CN(C2OC(COP(=O)(O)OP(=O)(O)OCC3OC(n4c...,ACFIXJIJDZMPPO,C21H30N7O17P3,745.091102,C21H30N7O17P3



Adduct counts in validation queries:


adduct
[M+H]+           954
[M-H]-           407
[M+Na]+          166
[2M+Na]+         129
[M+CH2O2-H]-      78
[2M+H]+           77
[M+NH4]+          58
[2M-H]-           54
[M+Cl]-           24
[M-H2O+H]+        22
[M+K]+            15
[M+C2H4O2-H]-      4
[M]+               3
[M-2H2O+H]+        2
[2M+K]+            1
Name: count, dtype: int64

In [125]:

# Check how well each adduct equation predicts the known
# structure's observed precursor m/z.

from collections import Counter
import numpy as np
import pandas as pd

mass_check = queries[
    ["inchikey14", "precursor_mz", "adduct"]
].merge(
    library[["inchikey14", "exact_mass"]],
    on="inchikey14",
    how="left"
)

mass_check["predicted_mz"] = [
    expected_mz(mass, adduct)
    if pd.notna(mass) else np.nan
    for mass, adduct in zip(
        mass_check["exact_mass"],
        mass_check["adduct"]
    )
]

mass_check["abs_error_ppm"] = (
    (mass_check["precursor_mz"] - mass_check["predicted_mz"])
    / mass_check["predicted_mz"]
    * 1e6
).abs()

mass_check["valid"] = (
    np.isfinite(mass_check["predicted_mz"])
    & (mass_check["predicted_mz"] > 0)
    & np.isfinite(mass_check["abs_error_ppm"])
)

audit = (
    mass_check.groupby("adduct", dropna=False)
    .agg(
        spectra=("inchikey14", "size"),
        valid_equations=("valid", "sum"),
        median_abs_ppm=("abs_error_ppm", "median"),
        pct_within_10ppm=(
            "abs_error_ppm",
            lambda x: (x <= 10).mean() * 100
        )
    )
    .sort_values("spectra", ascending=False)
)

display(audit.round(2))
print("\nAdducts with no valid equation:")
print(audit.index[audit["valid_equations"] == 0].tolist())


,spectra,valid_equations,median_abs_ppm,pct_within_10ppm
adduct,,,,
[M+H]+,954,954,0.98,91.40
[M-H]-,407,407,0.17,98.53
[M+Na]+,166,166,3.14,77.71
[2M+Na]+,129,129,1.07,100.00
[M+CH2O2-H]-,78,78,0.07,100.00
[2M+H]+,77,77,1.70,100.00
[M+NH4]+,58,58,0.10,96.55
[2M-H]-,54,54,0.81,100.00
[M+Cl]-,24,24,0.07,100.00



Adducts with no valid equation:
['[M+C2H4O2-H]-', '[2M+Na-2H]-', '[2M+K]+', '[M-H2]2-']


In [129]:

# Inspect adducts that expected_mz() does not currently support.

unsupported_adducts = [
    "[M+C2H4O2-H]-",
    "[2M+Na-2H]-",
    "[2M+K]+",
    "[M-H2]2-",
]

unsupported_cases = mass_check[
    mass_check["adduct"].isin(unsupported_adducts)
][
    [
        "inchikey14",
        "adduct",
        "precursor_mz",
        "exact_mass",
        "normalized_smiles",
    ]
].copy()

# Add the true molecular formula for chemical interpretation.
unsupported_cases = unsupported_cases.merge(
    library[["inchikey14", "molecular_formula"]],
    on="inchikey14",
    how="left"
)

display(unsupported_cases.to_string(index=False))


KeyError: "['normalized_smiles'] not in index"

In [128]:

unsupported_adducts = [
    "[M+C2H4O2-H]-",
    "[2M+Na-2H]-",
    "[2M+K]+",
    "[M-H2]2-",
]

unsupported_cases = mass_check[
    mass_check["adduct"].isin(unsupported_adducts)
][
    [
        "inchikey14",
        "adduct",
        "precursor_mz",
        "exact_mass",
    ]
].merge(
    library[
        ["inchikey14", "molecular_formula", "normalized_smiles"]
    ],
    on="inchikey14",
    how="left"
)

display(unsupported_cases.to_string(index=False))


'    inchikey14        adduct  precursor_mz  exact_mass molecular_formula                                                               normalized_smiles\nXVPZRKIQCKKYNE       [2M+K]+      539.1840  250.110613         C16H14N2O                                                  Cn1c(Cc2ccccc2)nc(=O)c2ccccc21\nWZBDRAAGHRCRKM   [2M+Na-2H]-      513.1330  246.079313         C16H10N2O                                                     Oc1cc2nc3ccccc3nc2c2ccccc12\nPTKWWHCMNOQBSL [M+C2H4O2-H]-      854.6281  795.614191        C46H86NO7P CCC=CCC=CCC=CCC=CCCCCCCC(=O)OC(COCCCCCCCCCCCCCCCCCC)COP(=O)([O-])OCC[N+](C)(C)C\nIDRKESNUEGURSD [M+C2H4O2-H]-      608.3900  549.379440        C28H56NO7P                      CCCCCCCCCCCCCCCC=CCCC(=O)OC(CO)COP(=O)([O-])OCC[N+](C)(C)C\nOFBHPPMPBOJXRT      [M-H2]2-      230.5000  463.074043      C14H18N5O11P                          O=C(O)CC(Nc1ncnc2c1ncn2C1OC(COP(=O)(O)O)C(O)C1O)C(=O)O\nWMCPHMOQTITNBG [M+C2H4O2-H]-      860.6386  801.624755        C45H88NO8P CC

In [130]:

# Add explicit support for the missing adduct labels.
# Keep the existing expected_mz() function intact for now.

POTASSIUM_CATION = 38.963158
SODIUM_CATION = 22.989218
PROTON_MASS = 1.0072764666
HYDROGEN_ATOM = 1.0078250322
ACETIC_ACID = 60.021129

def expected_mz_extended(neutral_mass, adduct):
    if adduct == "[2M+K]+":
        return 2 * neutral_mass + POTASSIUM_CATION

    if adduct == "[2M+Na-2H]-":
        return 2 * neutral_mass + SODIUM_CATION - 2 * PROTON_MASS

    if adduct == "[M+C2H4O2-H]-":
        return neutral_mass + ACETIC_ACID - PROTON_MASS

    if adduct == "[M-H2]2-":
        return (neutral_mass - 2 * HYDROGEN_ATOM) / 2

    return expected_mz(neutral_mass, adduct)


# Verify the new equations on the seven observed cases.
unsupported_cases["predicted_mz_new"] = [
    expected_mz_extended(mass, adduct)
    for mass, adduct in zip(
        unsupported_cases["exact_mass"],
        unsupported_cases["adduct"]
    )
]

unsupported_cases["abs_error_ppm_new"] = (
    (
        unsupported_cases["precursor_mz"]
        - unsupported_cases["predicted_mz_new"]
    )
    / unsupported_cases["predicted_mz_new"]
    * 1e6
).abs()

display(
    unsupported_cases[
        [
            "adduct",
            "precursor_mz",
            "predicted_mz_new",
            "abs_error_ppm_new",
        ]
    ].round(3)
)


,adduct,precursor_mz,predicted_mz_new,abs_error_ppm_new
0,[2M+K]+,539.184,539.184,0.712
1,[2M+Na-2H]-,513.133,513.133,0.567
2,[M+C2H4O2-H]-,854.628,854.628,0.066
3,[M+C2H4O2-H]-,608.390,608.393,5.411
4,[M-H2]2-,230.500,230.529,126.650
5,[M+C2H4O2-H]-,860.639,860.639,0.009
6,[M+C2H4O2-H]-,830.590,830.592,1.996


In [131]:

# Inspect the original training records for the [M-H2]2- case.
target_key = "OFBHPPMPBOJXRT"

train_path = "/kaggle/input/datasets/denisketer/enveda-casmi-2026/train.parquet"

import pyarrow.parquet as pq

pf = pq.ParquetFile(train_path)
matches = []

for batch in pf.iter_batches(
    columns=[
        "inchikey14",
        "normalized_smiles",
        "molecular_formula",
        "adduct",
        "precursor_mz",
        "precursor_error_ppm",
    ],
    batch_size=100_000,
):
    frame = batch.to_pandas()
    found = frame[
        (frame["inchikey14"] == target_key)
        & (frame["adduct"] == "[M-H2]2-")
    ]
    if not found.empty:
        matches.append(found)

if matches:
    display(pd.concat(matches, ignore_index=True).head(20))
else:
    print("No matching original training records found.")


,inchikey14,normalized_smiles,molecular_formula,adduct,precursor_mz,precursor_error_ppm
0,OFBHPPMPBOJXRT,O=C(O)CC(Nc1ncnc2c1ncn2C1OC(COP(=O)(O)O)C(O)C1...,C14H18N5O11P,[M-H2]2-,230.5,126.666706
1,OFBHPPMPBOJXRT,O=C(O)CC(Nc1ncnc2c1ncn2C1OC(COP(=O)(O)O)C(O)C1...,C14H18N5O11P,[M-H2]2-,230.5,126.666706
2,OFBHPPMPBOJXRT,O=C(O)CC(Nc1ncnc2c1ncn2C1OC(COP(=O)(O)O)C(O)C1...,C14H18N5O11P,[M-H2]2-,230.5,126.666706
3,OFBHPPMPBOJXRT,O=C(O)CC(Nc1ncnc2c1ncn2C1OC(COP(=O)(O)O)C(O)C1...,C14H18N5O11P,[M-H2]2-,230.5,126.666706
4,OFBHPPMPBOJXRT,O=C(O)CC(Nc1ncnc2c1ncn2C1OC(COP(=O)(O)O)C(O)C1...,C14H18N5O11P,[M-H2]2-,230.5,126.666706
5,OFBHPPMPBOJXRT,O=C(O)CC(Nc1ncnc2c1ncn2C1OC(COP(=O)(O)O)C(O)C1...,C14H18N5O11P,[M-H2]2-,230.5,126.666706
6,OFBHPPMPBOJXRT,O=C(O)CC(Nc1ncnc2c1ncn2C1OC(COP(=O)(O)O)C(O)C1...,C14H18N5O11P,[M-H2]2-,230.5,126.666706
7,OFBHPPMPBOJXRT,O=C(O)CC(Nc1ncnc2c1ncn2C1OC(COP(=O)(O)O)C(O)C1...,C14H18N5O11P,[M-H2]2-,230.5,126.666706
8,OFBHPPMPBOJXRT,O=C(O)CC(Nc1ncnc2c1ncn2C1OC(COP(=O)(O)O)C(O)C1...,C14H18N5O11P,[M-H2]2-,230.5,126.666706
9,OFBHPPMPBOJXRT,O=C(O)CC(Nc1ncnc2c1ncn2C1OC(COP(=O)(O)O)C(O)C1...,C14H18N5O11P,[M-H2]2-,230.5,126.666706


In [132]:
target_key = "OFBHPPMPBOJXRT"

cols = [
    "inchikey14",
    "adduct",
    "precursor_mz",
    "precursor_error_ppm",
    "adduct_orig",
    "ingest_lib",
    "collision_energy_orig",
    "collision_energy_orig_units",
]

matches = []

for batch in pf.iter_batches(columns=cols, batch_size=100_000):
    frame = batch.to_pandas()
    found = frame[
        (frame["inchikey14"] == target_key)
        & (frame["adduct"] == "[M-H2]2-")
    ]
    if not found.empty:
        matches.append(found)

if matches:
    diagnostic = pd.concat(matches, ignore_index=True)
    print("Matching rows:", len(diagnostic))
    display(diagnostic.drop_duplicates().head(30))
    print("\nUnique values per column:")
    display(diagnostic.nunique(dropna=False).to_frame("unique_values"))
else:
    print("No matching rows found.")

Matching rows: 13


,inchikey14,adduct,precursor_mz,precursor_error_ppm,adduct_orig,ingest_lib,collision_energy_orig,collision_energy_orig_units
0,OFBHPPMPBOJXRT,[M-H2]2-,230.5,126.666706,[M-H2]2-,massbank,30 V,V
1,OFBHPPMPBOJXRT,[M-H2]2-,230.5,126.666706,[M-H2]2-,massbank,50 V,V
2,OFBHPPMPBOJXRT,[M-H2]2-,230.5,126.666706,[M-H2]2-,massbank,40 V,V
3,OFBHPPMPBOJXRT,[M-H2]2-,230.5,126.666706,[M-H2]2-,massbank,20 V,V
4,OFBHPPMPBOJXRT,[M-H2]2-,230.5,126.666706,[M-H2]2-,massbank,10 V,V
5,OFBHPPMPBOJXRT,[M-H2]2-,230.5,126.666706,[M-H2]2-,mona,50 V,V
6,OFBHPPMPBOJXRT,[M-H2]2-,230.5,126.666706,[M-H2]2-,mona,40 V,V
7,OFBHPPMPBOJXRT,[M-H2]2-,230.5,126.666706,[M-H2]2-,mona,10 V,V
8,OFBHPPMPBOJXRT,[M-H2]2-,230.5,126.666706,[M-H2]2-,riken,30 V,V
9,OFBHPPMPBOJXRT,[M-H2]2-,230.5,126.666706,[M-H2]2-,riken,20 V,V



Unique values per column:


,unique_values
inchikey14,1
adduct,1
precursor_mz,1
precursor_error_ppm,2
adduct_orig,1
ingest_lib,3
collision_energy_orig,5
collision_energy_orig_units,1


In [133]:
display(
    diagnostic.groupby(
        ["precursor_error_ppm", "ingest_lib"],
        dropna=False
    )
    .size()
    .reset_index(name="row_count")
    .sort_values("precursor_error_ppm")
)

,precursor_error_ppm,ingest_lib,row_count
0,126.666706,massbank,5
1,126.666706,mona,3
2,126.666706,riken,5


In [134]:
observed_mz = 230.5
reported_error_ppm = 126.666706

# If ppm error is relative to the theoretical m/z:
theoretical_mz = observed_mz / (1 - reported_error_ppm / 1_000_000)

print(f"Observed m/z:              {observed_mz:.6f}")
print(f"Theoretical m/z implied:   {theoretical_mz:.6f}")
print(f"Our current predicted m/z: {expected_mz_extended(463.074043, '[M-H2]2-'):.6f}")

Observed m/z:              230.500000
Theoretical m/z implied:   230.529200
Our current predicted m/z: 230.529196


In [135]:
# Summarize large precursor errors by adduct.
# Use the original training parquet data, not the candidate library.

error_summary = (
    train_meta.assign(
        abs_error_ppm=train_meta["precursor_error_ppm"].abs()
    )
    .groupby("adduct")
    .agg(
        spectra=("abs_error_ppm", "size"),
        over_50_ppm=("abs_error_ppm", lambda x: (x > 50).sum()),
        over_100_ppm=("abs_error_ppm", lambda x: (x > 100).sum()),
        median_abs_ppm=("abs_error_ppm", "median"),
    )
    .sort_values("over_100_ppm", ascending=False)
)

display(error_summary.head(15))

,spectra,over_50_ppm,over_100_ppm,median_abs_ppm
adduct,,,,
[M+CH2O2-H]-,83188,52497,52484,1990.065333
[M+H]+,1324585,24106,23126,1.370212
[M-H]-,465688,12964,11552,1.874744
[M+Na]+,178092,4695,4360,2.715026
[M+NH4]+,53452,1788,1728,0.914964
[M-H2O+H]+,21682,1569,1502,2.069005
[M+K]+,18955,1019,986,1.324134
[M-2H2O+H]+,3439,609,609,1.723178
[M]+,8818,541,526,1.674876


In [136]:
import numpy as np
import pandas as pd

adducts_to_check = ["[M+CH2O2-H]-", "[M-H]-"]

for adduct in adducts_to_check:
    errors = train_meta.loc[
        train_meta["adduct"].eq(adduct),
        "precursor_error_ppm"
    ].dropna().abs()

    print(f"\n{adduct} — n={len(errors):,}")
    print(
        errors.quantile(
            [0, 0.5, 0.9, 0.95, 0.99, 0.999, 1]
        ).to_string()
    )
    print(f"Over 10 ppm:  {(errors > 10).mean():.2%}")
    print(f"Over 50 ppm:  {(errors > 50).mean():.2%}")
    print(f"Over 100 ppm: {(errors > 100).mean():.2%}")


[M+CH2O2-H]- — n=83,184
0.000         0.000242
0.500      1990.065333
0.900      2796.792287
0.950      2909.712743
0.990      3378.888771
0.999      5111.329770
1.000    131016.028191
Over 10 ppm:  63.15%
Over 50 ppm:  63.11%
Over 100 ppm: 63.09%

[M-H]- — n=464,497
0.000    3.203341e-05
0.500    1.874744e+00
0.900    3.645669e+00
0.950    4.570204e+00
0.990    7.220478e+02
0.999    1.686620e+05
1.000    2.590672e+06
Over 10 ppm:  3.34%
Over 50 ppm:  2.79%
Over 100 ppm: 2.49%


In [137]:
# Compare formate-adduct mass errors by source library.
formate = train_meta.loc[
    train_meta["adduct"].eq("[M+CH2O2-H]-"),
    ["inchikey14", "adduct", "precursor_mz",
     "precursor_error_ppm"]
].copy()

# Recover the source-library label from the original parquet.
source_rows = []

for batch in pf.iter_batches(
    columns=["inchikey14", "adduct", "precursor_mz",
             "precursor_error_ppm", "ingest_lib"],
    batch_size=100_000
):
    part = batch.to_pandas()
    part = part[part["adduct"].eq("[M+CH2O2-H]-")]
    if not part.empty:
        source_rows.append(part)

formate_sources = pd.concat(source_rows, ignore_index=True)
formate_sources["abs_error_ppm"] = (
    formate_sources["precursor_error_ppm"].abs()
)

summary = (
    formate_sources.groupby("ingest_lib")["abs_error_ppm"]
    .agg(
        spectra="size",
        median="median",
        p90=lambda x: x.quantile(0.90),
        over_100ppm=lambda x: (x > 100).mean()
    )
    .sort_values("over_100ppm", ascending=False)
)

display(summary)

,spectra,median,p90,over_100ppm
ingest_lib,,,,
pluskal_ms2,50785,2414.522823,2876.419510,1.000000
gnps,3321,691.407312,2529.888622,0.505269
mona,700,0.823488,2.135824,0.025714
riken,1556,0.765526,1.966790,0.001285
massbank,1095,0.860946,1.788645,0.000913
enveda-180,24556,1.735924,2.906515,0.000000
enveda-np-examples,105,0.789362,2.360931,0.000000
msdial,257,0.860949,1.396836,0.000000
spectraverse,813,0.685962,1.517570,0.000000


In [138]:
# Sample formate-adduct records from the two contrasting sources.
sample_rows = (
    formate_sources[
        formate_sources["ingest_lib"].isin(["pluskal_ms2", "mona"])
    ]
    .groupby("ingest_lib", group_keys=False)
    .sample(n=5, random_state=42)
    .copy()
)

sample_rows = sample_rows.merge(
    library[["inchikey14", "exact_mass"]].drop_duplicates("inchikey14"),
    on="inchikey14",
    how="left"
)

sample_rows["calculated_mz"] = sample_rows.apply(
    lambda row: expected_mz_extended(
        row["exact_mass"], row["adduct"]
    ),
    axis=1
)

sample_rows["recomputed_error_ppm"] = (
    (sample_rows["precursor_mz"] - sample_rows["calculated_mz"])
    / sample_rows["calculated_mz"] * 1_000_000
)

display(
    sample_rows[
        [
            "ingest_lib", "inchikey14", "precursor_mz",
            "exact_mass", "calculated_mz",
            "precursor_error_ppm", "recomputed_error_ppm"
        ]
    ].sort_values("ingest_lib")
)

,ingest_lib,inchikey14,precursor_mz,exact_mass,calculated_mz,precursor_error_ppm,recomputed_error_ppm
0,mona,GOYZKWCPWBKPIG,531.2241,486.225368,531.223569,2.028666,0.999489
1,mona,RXAXTTGJEMODPY,625.1410,580.142820,625.141021,0.840663,-0.033906
2,mona,OZDABLANSWPSGY,655.2240,610.226156,655.224357,0.289718,-0.544693
3,mona,GOLVEYNUJMICCU,581.2240,536.225762,581.223963,1.004351,0.063707
4,mona,MHWAHKIDOYBXCN,667.2390,622.241412,667.239613,0.099382,-0.918764
5,pluskal_ms2,TVVBYYQCNUJOPR,425.1631,380.164854,425.163055,2369.146087,0.105531
6,pluskal_ms2,VUMZHZYKXUYIHM,413.2181,368.219889,413.218090,2437.642680,0.024820
7,pluskal_ms2,BIZIKZVVSNWJGL,347.1646,302.166414,347.164615,2901.428099,-0.042320
8,pluskal_ms2,HKMWINBOLFNBIJ,386.1721,341.173942,386.172143,2608.372828,-0.110303
9,pluskal_ms2,NUBWFWVVKLRSHS,441.1831,396.184921,441.183122,2283.133905,-0.049594


In [139]:
# Compare stored versus independently recomputed ppm error
# for a random sample of formate-adduct records.

check = (
    formate_sources
    .groupby("ingest_lib", group_keys=False)
    .apply(
        lambda g: g.sample(
            n=min(500, len(g)),
            random_state=42
        ),
        include_groups=True
    )
    .reset_index(drop=True)
)

check = check.merge(
    library[["inchikey14", "exact_mass"]].drop_duplicates("inchikey14"),
    on="inchikey14",
    how="left"
)

check["calculated_mz"] = check.apply(
    lambda r: expected_mz_extended(r["exact_mass"], r["adduct"]),
    axis=1
)

check["recomputed_error_ppm"] = (
    (check["precursor_mz"] - check["calculated_mz"])
    / check["calculated_mz"] * 1_000_000
)

check["stored_abs_error"] = check["precursor_error_ppm"].abs()
check["recomputed_abs_error"] = check["recomputed_error_ppm"].abs()

display(
    check.groupby("ingest_lib").agg(
        sampled_rows=("inchikey14", "size"),
        stored_median_abs_ppm=("stored_abs_error", "median"),
        recomputed_median_abs_ppm=("recomputed_abs_error", "median"),
        recomputed_pct_within_10ppm=(
            "recomputed_abs_error",
            lambda x: (x <= 10).mean() * 100
        )
    ).sort_values("sampled_rows", ascending=False)
)

/tmp/ipykernel_48/722207802.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


,sampled_rows,stored_median_abs_ppm,recomputed_median_abs_ppm,recomputed_pct_within_10ppm
ingest_lib,,,,
enveda-180,500,1.794393,0.683896,100.0
gnps,500,706.817640,0.694702,100.0
massbank,500,0.860946,0.035182,99.2
mona,500,0.789741,0.476029,96.0
pluskal_ms2,500,2347.182668,0.056951,100.0
spectraverse,500,0.685565,0.055074,100.0
riken,500,0.758627,0.301969,98.6
msdial,257,0.860949,0.034492,100.0
enveda-np-examples,105,0.789362,0.793268,100.0


In [140]:
# Sample up to 300 records per common adduct and source.
# Recompute ppm errors independently of precursor_error_ppm.

common_adducts = ["[M+H]+", "[M-H]-", "[M+Na]+", "[M+CH2O2-H]-"]

check_common = []

for batch in pf.iter_batches(
    columns=["inchikey14", "adduct", "precursor_mz", "ingest_lib"],
    batch_size=100_000
):
    part = batch.to_pandas()
    part = part[part["adduct"].isin(common_adducts)]
    if not part.empty:
        check_common.append(part)

check_common = pd.concat(check_common, ignore_index=True)

check_common = (
    check_common
    .groupby(["adduct", "ingest_lib"], group_keys=False)
    .apply(
        lambda g: g.sample(n=min(300, len(g)), random_state=42)
    )
    .reset_index(drop=True)
)

check_common = check_common.merge(
    library[["inchikey14", "exact_mass"]].drop_duplicates("inchikey14"),
    on="inchikey14",
    how="left"
)

check_common["calculated_mz"] = check_common.apply(
    lambda r: expected_mz_extended(r["exact_mass"], r["adduct"]),
    axis=1
)

check_common["abs_recomputed_ppm"] = (
    (check_common["precursor_mz"] - check_common["calculated_mz"]).abs()
    / check_common["calculated_mz"] * 1_000_000
)

display(
    check_common.groupby(["adduct", "ingest_lib"]).agg(
        n=("inchikey14", "size"),
        median_abs_ppm=("abs_recomputed_ppm", "median"),
        pct_within_10ppm=(
            "abs_recomputed_ppm",
            lambda x: (x <= 10).mean() * 100
        )
    ).sort_values(["adduct", "pct_within_10ppm"])
)

/tmp/ipykernel_48/2698977846.py:22: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


n  median_abs_ppm  pct_within_10ppm
adduct       ingest_lib                                               
[M+CH2O2-H]- mona                300        0.492409         95.333333
             massbank            300        0.035182         98.666667
             riken               300        0.303983         98.666667
             enveda-180          300        0.715448        100.000000
             enveda-np-examples  105        0.793268        100.000000
             gnps                300        0.692581        100.000000
             msdial              257        0.034492        100.000000
             pluskal_ms2         300        0.058134        100.000000
             spectraverse        300        0.054309        100.000000
[M+H]+       riken               300        3.532552         75.333333
             gnps                300        1.033576         91.666667
             massbank            300        0.104512         91.666667
             mona                300        0.124266         94.000000
             drug_plus           300        0.080993         99.666667
             enveda-180          300        2.246762        100.000000
             enveda-np-examples  300        2.004559        100.000000
             masaryk             300        0.592535        100.000000
             msdial              300        0.080577        100.000000
             pluskal_ms2         300        0.074825        100.000000
             spectraverse        300        0.462084        100.000000
[M+Na]+      riken               300        5.577387         72.333333
             drug_plus           225        0.070530         79.111111
             mona                300        0.094909         96.000000
             gnps                300        0.758482         97.000000
             massbank            300        0.078323         99.666667
             enveda-180          300        1.302833        100.000000
             enveda-np-examples   80        1.703627        100.000000
             msdial              300        0.041801        100.000000
             pluskal_ms2         300        0.070586        100.000000
             spectraverse        300        0.464626        100.000000
[M-H]-       riken               300        0.173692         74.333333
             massbank            300        0.105906         87.666667
             mona                300        0.128843         90.000000
             gnps                300        0.800883         98.333333
             enveda-np-examples  276        0.944516         99.637681
             enveda-180          300        0.803051        100.000000
             msdial              300        0.065676        100.000000
             pluskal_ms2         300        0.067388        100.000000
             spectraverse        300        0.228012        100.000000

In [141]:

from pathlib import Path
from datetime import datetime

checkpoint = r"""
ENVEDA CASMI 2026 — PROJECT CHECKPOINT
Purpose: Resume the Kaggle molecule identification project.

DATA
- Dataset: /kaggle/input/datasets/denisketer/enveda-casmi-2026/
- Files: train.parquet, test.parquet, sample_submission.csv
- Train: 2,539,608 spectra rows
- Test: 1,213 spectra across 400 molecule IDs
- Candidate library: 275,810 unique InChIKey14 structures

CURRENT APPROACH
- Build candidate library from training structures using RDKit.
- Use molecular exact mass and adduct information for candidate filtering.
- Use MS/MS fragment similarity to rank candidates.
- Produce up to 25 semicolon-separated SMILES per molecule_id.
- Evaluation target: MRR@25 after tautomer canonicalization.

IMPORTANT NOTEBOOK OBJECTS
- library: 275,810 rows; normalized_smiles, inchikey14,
  molecular_formula, exact_mass, rdkit_formula
- train_meta: precursor/adduct metadata from training data
- test_meta: test molecule and precursor metadata
- refs_all: one reference spectrum per structure
- queries: 2,000 sampled validation spectra
- Q and R: sparse query/reference fragment-feature matrices
- expected_mz and expected_mz_extended: adduct mass functions
- submission_baseline.csv: format smoke test only, not a competitive model

VALIDATION WORK
- Mass-only retrieval needs adduct-aware mass calculations.
- Preliminary fragment retrieval is a diagnostic, not competition performance.
- Do not interpret earlier biased or preliminary benchmark scores as final results.
- The stored precursor_error_ppm can be unreliable for some source libraries.
- For [M+CH2O2-H]-, pluskal_ms2 and gnps have very large stored errors,
  but sampled records have small independently recomputed mass errors.
- Do not use stored precursor_error_ppm as a hard candidate filter.
- [M-H2]2- has an unresolved observed/theoretical mass discrepancy.

NEXT STEP
1. Inspect independently recomputed mass errors for common adducts:
   [M+H]+, [M-H]-, [M+Na]+, [M+CH2O2-H]-.
2. Then decide evidence-based mass tolerances and fallback handling.
3. Validate candidate retrieval before building the final submission.

SAFETY
- Preserve the current notebook and do not overwrite the original train data.
- Recheck variable availability after restarting the session.
- Recreate objects from the original parquet files if necessary.
"""

path = Path("/kaggle/working/progress_checkpoint.txt")
path.write_text(
    f"Checkpoint saved: {datetime.now().isoformat(timespec='minutes')}\n"
    + checkpoint,
    encoding="utf-8"
)

print("Checkpoint saved:", path)
print("File size:", path.stat().st_size, "bytes")


Checkpoint saved: /kaggle/working/progress_checkpoint.txt
File size: 2336 bytes
